# Polymarket, MOMENT y LoRA en Colab

Este archivo contiene el código y las versiones de las bibliotecas. Ábrelo directamente en Colab, selecciona una GPU como acelerador y ejecuta las celdas en orden. No necesitas clonar el repositorio ni subir scripts. El notebook descarga los datos de Polymarket y los pesos originales de MOMENT cuando hacen falta.

El modo **piloto** ejecuta el recorrido completo sobre el archivo del 6 de junio: descarga, comprobación, series, catálogo, ventanas, entrenamiento con recuperación, validación y prueba. Sus cortes son exploratorios y corresponden a ese día.

El modo **integral** cubre los 164 originales y aplica los cortes cronológicos del proyecto. La corrida local tomó unas 67 horas acumuladas dentro de sus intentos y guardó unos 765 GB solo en series. Una sesión habitual de Colab no ofrece espacio o duración garantizados para ese modo. Usa almacenamiento persistente con espacio suficiente y un sistema de archivos Linux con bloqueo y sincronización. El montaje habitual de Google Drive no está validado para las bases de datos y escrituras concurrentes del recorrido integral. El piloto puede ejecutarse en el disco temporal de Colab. Copia sus resultados antes de cerrar la sesión.

In [ ]:
from pathlib import Path

MODO = 'piloto'  # 'piloto' o 'integral'
RANGO_LORA = 8  # Cambia a 16 para probar un adaptador de mayor tamaño.
HORAS_PRESUPUESTO = 120
AUTORIZAR_INTEGRAL = False  # Actívalo solo con almacenamiento y tiempo suficientes.

# Cambia esta ruta para utilizar un disco persistente conectado al entorno.
BASE_TRABAJO = Path('/content/polymarket_colab')
PROYECTO = BASE_TRABAJO / f'{MODO}_r{RANGO_LORA}'
RESULTADOS = PROYECTO / 'runs/mvp'

if MODO not in ('piloto', 'integral') or type(RANGO_LORA) is not int or RANGO_LORA < 1:
    raise ValueError('Elige un modo válido y un rango LoRA entero positivo.')
if MODO == 'integral' and not AUTORIZAR_INTEGRAL:
    raise ValueError('Confirma el modo integral en la configuración.')

## Preparar los archivos internos

Las siguientes celdas escriben los módulos incluidos en este mismo notebook en la carpeta de trabajo. Se guardan como archivos Python porque algunos pasos inician procesos separados. La configuración de LoRA adapta las mismas capas del MVP y usa una escala constante al cambiar el rango.

In [ ]:
import base64
import hashlib
import shutil
import subprocess
import sys
import zlib

def guardar(ruta, contenido):
    destino = PROYECTO / ruta
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists() or destino.read_bytes() != contenido.encode('utf-8'):
        destino.write_text(contenido, encoding='utf-8')

def ajustar_modelo(contenido):
    original = 'r=8,\n            lora_alpha=16,'
    if contenido.count(original) != 1:
        raise ValueError('No se encontró la configuración LoRA esperada.')
    return contenido.replace(
        original, f'r={RANGO_LORA},\n            lora_alpha={2 * RANGO_LORA},'
    )

def ajustar_motor(contenido):
    if MODO == 'piloto':
        # El piloto solo necesita guardar los derivados de un archivo.
        original = 'shutil.disk_usage(root).free < 100 * 1024**3'
        if contenido.count(original) != 1:
            raise ValueError('No se encontró el control de espacio esperado.')
        return contenido.replace(original, 'shutil.disk_usage(root).free < 8 * 1024**3')
    return contenido

In [ ]:
def modulo(ruta, contenido):
    ruta = ruta.strip()
    if ruta.endswith('/model.py'):
        contenido = ajustar_modelo(contenido)
    elif ruta.endswith('/engine.py'):
        contenido = ajustar_motor(contenido)
    guardar(ruta, contenido.rstrip('\n') + '\n')

get_ipython().register_magic_function(modulo, 'cell', 'modulo')

**src/polymarket_mvp/__init__.py**

In [ ]:
%%modulo src/polymarket_mvp/__init__.py
"""Single-pass original-event processing and auditable MOMENT experiments."""

REVISION = 'efe472f1f00a62f2cab1fed2851da439d9c1fe11'
MODEL_REVISION = '411e288267f82cce86296dbe4d6c8bc533cc162f'
REPOSITORY = 'Joseph3222/polymarket-orderbook'

**src/polymarket_mvp/catalog.py**

In [ ]:
%%modulo src/polymarket_mvp/catalog.py
"""Post-ingest catalog audit; never a dependency for producing early features."""

from pathlib import Path
import json, os, hashlib
import duckdb
from .storage import Manifest
from .download import atomic_json


def build_catalog(partition_roots, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    paths = []
    for root in partition_roots:
        root = Path(root)
        m = Manifest(root / 'parts')
        paths.extend(
            str((root / 'parts' / r[0]).resolve())
            for r in m.db.execute(
                "SELECT path FROM objects WHERE path LIKE '%_relations.parquet'"
            )
        )
        m.db.close()
    identity = hashlib.sha256(json.dumps(sorted(paths)).encode()).hexdigest()
    staging = output / 'catalog_work.duckdb'
    db = duckdb.connect(str(staging))
    db.execute("SET memory_limit='8GB'")
    db.execute("SET threads=2")
    db.execute("SET preserve_insertion_order=false")
    db.execute('SET temp_directory=?', [str(output / 'spill')])
    progress = output / 'progress.json'
    previous = json.loads(progress.read_text()) if progress.exists() else {}
    tables = {row[0] for row in db.execute("SHOW TABLES").fetchall()}
    imported = (
        'relations' in tables
        and previous.get('source_files_confirmed') == len(paths)
        and previous.get('source_files') == len(paths)
        and previous.get('input_identity', identity) == identity
    )
    if not imported:
        db.execute('DROP TABLE IF EXISTS relations')
        for offset in range(0, len(paths), 128):
            db.from_parquet(paths[offset : offset + 128]).create_view(
                'relation_batch', replace=True
            )
            if offset == 0:
                db.execute('CREATE TABLE relations AS SELECT * FROM relation_batch')
            else:
                db.execute('INSERT INTO relations SELECT * FROM relation_batch')
            atomic_json(
                progress,
                dict(
                    source_files_confirmed=min(offset + 128, len(paths)),
                    source_files=len(paths),
                    input_identity=identity,
                    pid=os.getpid(),
                ),
            )
    db.execute('CHECKPOINT')
    atomic_json(
        progress,
        dict(
            source_files_confirmed=len(paths),
            source_files=len(paths),
            input_identity=identity,
            pid=os.getpid(),
            state='aggregating',
            reused_import=imported,
        ),
    )
    q = '''SELECT asset_id,condition_id,min(first_ts) AS first_observed_ms,max(last_ts) AS last_observed_ms,
         sum(events)::UBIGINT AS original_events FROM relations GROUP BY asset_id,condition_id'''
    path = output / 'observed_relations.parquet'
    db.execute(f"COPY ({q}) TO '{path}' (FORMAT PARQUET,COMPRESSION ZSTD)").fetchall()
    db.from_parquet(str(path)).create_view('catalog')
    conflicts = output / 'multiple_observed_conditions.parquet'
    db.execute(
        f"COPY (SELECT asset_id,count(*) AS observed_conditions,min(first_observed_ms) AS first_ms,max(last_observed_ms) AS last_ms FROM catalog GROUP BY asset_id HAVING count(*)>1) TO '{conflicts}' (FORMAT PARQUET,COMPRESSION ZSTD)"
    ).fetchall()
    count = db.execute(
        'SELECT count(DISTINCT asset_id),count(DISTINCT condition_id),count(*) FROM catalog'
    ).fetchall()[0]
    r = dict(
        assets=count[0],
        conditions=count[1],
        observed_pairs=count[2],
        relation_derived_scan_passes=1,
        conflicts=db.execute(
            'SELECT count(*) FROM read_parquet(?)', [str(conflicts)]
        ).fetchall()[0][0],
        policy='Preserve reported historical IDs; exclude ambiguous asset/window identities; no global reassignment',
    )
    atomic_json(output / 'catalog.json', r)
    db.close()
    return r

**src/polymarket_mvp/columnar.py**

In [ ]:
%%modulo src/polymarket_mvp/columnar.py
"""Lossless Parquet encoding for monotone temporal and provenance integers."""

import pyarrow as pa
import pyarrow.parquet as pq

DELTA = {
    'second',
    'event_ts_ms_first',
    'event_ts_ms_last',
    'row_position_first',
    'row_position_last',
}


def options(schema, level=3):
    delta = {
        f.name: 'DELTA_BINARY_PACKED'
        for f in schema
        if f.name in DELTA and pa.types.is_integer(f.type)
    }
    return dict(
        compression='zstd',
        compression_level=level,
        use_dictionary=[f.name for f in schema if f.name not in delta],
        column_encoding=delta,
    )


def write(table, path, level=3):
    pq.write_table(table, path, **options(table.schema, level))

**src/polymarket_mvp/download.py**

In [ ]:
%%modulo src/polymarket_mvp/download.py
"""Pinned inventory and resumable HTTP staging with payload byte accounting."""

import hashlib
import json
import os
import time
from pathlib import Path
import httpx
from huggingface_hub import HfApi
from huggingface_hub.hf_api import RepoFile
from . import REVISION, REPOSITORY


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    with tmp.open('w') as f:
        json.dump(value, f, sort_keys=True, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)
    fd = os.open(path.parent, os.O_RDONLY | os.O_DIRECTORY)
    try:
        os.fsync(fd)
    finally:
        os.close(fd)


def inventory(root):
    path = Path(root) / 'inventory.json'
    if path.exists():
        data = json.loads(path.read_text())
    else:
        files = [
            f
            for f in HfApi().list_repo_tree(
                REPOSITORY,
                repo_type='dataset',
                revision=REVISION,
                path_in_repo='orderbook',
                recursive=True,
            )
            if isinstance(f, RepoFile) and f.path.endswith('.parquet')
        ]
        data = {
            'repository': REPOSITORY,
            'revision': REVISION,
            'files': [
                {
                    'path': f.path,
                    'bytes': f.size,
                    'oid': f.blob_id,
                    'sha256': getattr(f.lfs, 'sha256', None),
                }
                for f in sorted(files, key=lambda f: f.path)
            ],
        }
        for i, f in enumerate(data['files']):
            f['ordinal'] = i
        atomic_json(path, data)
    if (
        data['revision'] != REVISION
        or len(data['files']) != 164
        or sum(f['bytes'] for f in data['files']) != 1177799623341
    ):
        raise ValueError('Pinned inventory mismatch')
    return data


def download(entry, root, deadline=None):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    dest = root / (entry['path'].split('date=')[1][:10] + '.parquet')
    report = dest.with_suffix('.download.json')
    if dest.exists() and report.exists():
        r = json.loads(report.read_text())
        if (
            dest.stat().st_size == entry['bytes']
            and r['verified_sha256'] == entry['sha256']
        ):
            return dest, r
        raise ValueError('Staging identity mismatch')
    part = dest.with_suffix('.partial')
    if entry['bytes'] >= 1024**3:
        return parallel_download(entry, root, deadline=deadline)
    history = dest.with_suffix('.transfers.jsonl')
    start = time.perf_counter()
    transferred = 0
    url = f'https://huggingface.co/datasets/{REPOSITORY}/resolve/{REVISION}/{entry["path"]}'
    for attempt in range(12):
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError("Download budget reached; transfer fragments retained")
        offset = part.stat().st_size if part.exists() else 0
        got = 0
        t = time.perf_counter()
        error = None
        try:
            with httpx.stream(
                'GET',
                url,
                headers={'Range': f'bytes={offset}-', 'Accept-Encoding': 'identity'},
                follow_redirects=True,
                timeout=httpx.Timeout(120, connect=30),
            ) as response:
                response.raise_for_status()
                if offset and response.status_code != 206:
                    raise ValueError('Server ignored resume range')
                with part.open('ab' if offset else 'wb') as f:
                    for chunk in response.iter_bytes(8 << 20):
                        f.write(chunk)
                        got += len(chunk)
                    f.flush()
                    os.fsync(f.fileno())
        except Exception as exc:
            error = repr(exc)
        transferred += got
        with history.open('a') as f:
            f.write(
                json.dumps(
                    dict(
                        offset=offset,
                        bytes=got,
                        seconds=time.perf_counter() - t,
                        error=error,
                        attempt=attempt,
                        utc=time.time(),
                    )
                )
                + '\n'
            )
        if error is None and part.stat().st_size == entry['bytes']:
            break
        if error:
            print('DOWNLOAD_RETRY', entry['path'], error, flush=True)
        time.sleep(min(30, attempt + 1))
    else:
        raise RuntimeError('Download retries exhausted')
    t = time.perf_counter()
    h = hashlib.sha256()
    with part.open('rb') as f:
        for block in iter(lambda: f.read(16 << 20), b''):
            h.update(block)
    if h.hexdigest() != entry['sha256']:
        raise ValueError('Original SHA256 mismatch')
    os.replace(part, dest)
    records = [json.loads(line) for line in history.read_text().splitlines()]
    r = dict(
        path=entry['path'],
        revision=REVISION,
        original_bytes=entry['bytes'],
        payload_bytes=sum(x['bytes'] for x in records),
        network_seconds=sum(x['seconds'] for x in records),
        verification_seconds=time.perf_counter() - t,
        seconds=time.perf_counter() - start,
        requests=len(records),
        verified_sha256=h.hexdigest(),
    )
    atomic_json(report, r)
    return dest, r


def parallel_download(entry, root, workers=4, deadline=None):
    from concurrent.futures import ThreadPoolExecutor
    from threading import Lock

    root = Path(root)
    dest = root / (entry['path'].split('date=')[1][:10] + '.parquet')
    url = f'https://huggingface.co/datasets/{REPOSITORY}/resolve/{REVISION}/{entry["path"]}'
    size = entry['bytes']
    prefix = dest.with_suffix('.partial')
    prefix_size = prefix.stat().st_size if prefix.exists() else 0
    chunk = (size - prefix_size + workers - 1) // workers
    lock = Lock()
    history = dest.with_suffix('.parallel_transfers.jsonl')
    started = time.perf_counter()

    def fetch(i):
        first = prefix_size + i * chunk
        last = min(size, prefix_size + (i + 1) * chunk) - 1
        part = dest.with_suffix(f'.range{i}')
        for attempt in range(12):
            if deadline is not None and time.time() >= deadline:
                raise TimeoutError("Download budget reached; range fragments retained")
            offset = part.stat().st_size if part.exists() else 0
            if offset == last - first + 1:
                return part
            got = 0
            t = time.perf_counter()
            error = None
            try:
                with httpx.stream(
                    'GET',
                    url,
                    headers={
                        'Range': f'bytes={first+offset}-{last}',
                        'Accept-Encoding': 'identity',
                    },
                    follow_redirects=True,
                    timeout=httpx.Timeout(120, connect=30),
                ) as response:
                    response.raise_for_status()
                    if response.status_code != 206 or not response.headers.get(
                        'content-range', ''
                    ).startswith(f'bytes {first+offset}-'):
                        raise ValueError('Invalid range response')
                    with part.open('ab' if offset else 'wb') as f:
                        for data in response.iter_bytes(4 << 20):
                            f.write(data)
                            got += len(data)
                        f.flush()
                        os.fsync(f.fileno())
            except Exception as exc:
                error = repr(exc)
            with lock:
                with history.open('a') as f:
                    f.write(
                        json.dumps(
                            dict(
                                worker=i,
                                offset=first + offset,
                                bytes=got,
                                seconds=time.perf_counter() - t,
                                error=error,
                            )
                        )
                        + '\n'
                    )
            if error is None and part.stat().st_size == last - first + 1:
                return part
            time.sleep(min(30, attempt + 1))
        raise RuntimeError('Range transfer exhausted retries')

    with ThreadPoolExecutor(max_workers=workers) as pool:
        parts = list(pool.map(fetch, range(workers)))
    network = time.perf_counter() - started
    t = time.perf_counter()
    h = hashlib.sha256()
    tmp = dest.with_suffix('.assembled')
    with tmp.open('wb') as out:
        for part in ([prefix] if prefix_size else []) + parts:
            with part.open('rb') as f:
                for block in iter(lambda: f.read(16 << 20), b''):
                    out.write(block)
                    h.update(block)
        out.flush()
        os.fsync(out.fileno())
    if tmp.stat().st_size != size or h.hexdigest() != entry['sha256']:
        raise ValueError('Parallel download integrity failure')
    os.replace(tmp, dest)
    records = [json.loads(x) for x in history.read_text().splitlines()]
    r = dict(
        path=entry['path'],
        revision=REVISION,
        original_bytes=size,
        payload_bytes=prefix_size + sum(x['bytes'] for x in records),
        recovered_prefix_bytes=prefix_size,
        network_time_scope=(
            'current resumed attempt; prior prefix time excluded'
            if prefix_size
            else 'complete file transfer'
        ),
        prior_prefix_requests_unknown=bool(prefix_size),
        network_seconds=network,
        request_seconds_sum=sum(x['seconds'] for x in records),
        verification_seconds=time.perf_counter() - t,
        seconds=time.perf_counter() - started,
        requests=len(records),
        verified_sha256=h.hexdigest(),
        workers=workers,
    )
    atomic_json(dest.with_suffix('.download.json'), r)
    for part in parts:
        part.unlink()
    if prefix_size:
        prefix.unlink()
    return dest, r

**src/polymarket_mvp/engine.py**

In [ ]:
%%modulo src/polymarket_mvp/engine.py
"""Bounded-memory fused ingest, immutable parts, and externally spilled consolidation."""

import hashlib
import json
import os
import resource
import shutil
import time
from collections import Counter
from pathlib import Path

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import psutil

from .download import atomic_json
from .features import COLUMNS, KEYS, partial, merge
from .storage import Manifest
from . import REVISION, REPOSITORY

ENGINE_SOURCE = Path(__file__).read_text()
FEATURE_SOURCE = Path(__file__).with_name('features.py').read_text()


def usage():
    p = psutil.Process()
    io = p.io_counters()
    return dict(
        rss=p.memory_info().rss,
        peak_rss=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,
        cpu_seconds=sum(p.cpu_times()[:2]),
        disk_read_bytes=io.read_bytes,
        disk_write_bytes=io.write_bytes,
    )


# Lectura y extracción


def _process(
    entry,
    source,
    root,
    *,
    mode='local',
    batch_size=100288,
    groups_per_unit=64,
    max_groups=None,
    stream_groups=False,
    deadline=None,
    dictionary_read=False,
):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    m = Manifest(root / 'parts')
    run_meta = root / 'source.json'
    (root / 'engine_ingest_snapshot.py').write_text(ENGINE_SOURCE)
    (root / 'features_snapshot.py').write_text(FEATURE_SOURCE)
    code_hash = hashlib.sha256(FEATURE_SOURCE.encode()).hexdigest()
    identity = dict(
        revision=REVISION,
        source=entry,
        feature_code_sha256=code_hash,
        groups_per_unit=groups_per_unit,
    )
    if run_meta.exists() and json.loads(run_meta.read_text()) != identity:
        raise ValueError(
            'Source or feature code identity changed; use a new output directory'
        )
    atomic_json(run_meta, identity)
    m.verify()  # never trust a confirmed unit with missing or damaged objects
    started = time.perf_counter()
    before = usage()
    times = Counter()
    total = 0
    requests = []
    io_log = root / 'original_reads.jsonl'
    attempt = f'{os.getpid()}-{time.time_ns()}'
    if mode == 'remote':
        from huggingface_hub import HfFileSystem

        f = HfFileSystem().open(
            f'datasets/{REPOSITORY}@{REVISION}/' + entry['path'],
            'rb',
            block_size=4 << 20,
            cache_type='readahead',
        )
        fetch = f._fetch_range

        def counted(start, end):
            t = time.perf_counter()
            data = fetch(start, end)
            requests.append(
                dict(
                    start=start,
                    end=end,
                    bytes=len(data),
                    seconds=time.perf_counter() - t,
                )
            )
            return data

        f._fetch_range = counted
        f.cache.fetcher = (
            counted  # cache captured the original bound method during construction
        )
    else:
        f = open(source, 'rb')
    try:
        t = time.perf_counter()
        p = pq.ParquetFile(
            f,
            read_dictionary=(
                ['asset_id', 'condition_id', 'event_type', 'source_archive', 'side']
                if dictionary_read
                else None
            ),
        )
        times['footer_seconds'] += time.perf_counter() - t
        if set(COLUMNS) - set(p.schema_arrow.names):
            raise ValueError('Required columns missing')
        for name in COLUMNS:
            dtype = p.schema_arrow.field(name).type
            if name == 'event_ts_ms':
                valid = pa.types.is_integer(dtype)
            elif name in ['best_bid', 'best_ask', 'price', 'size']:
                valid = pa.types.is_floating(dtype) or pa.types.is_integer(dtype)
            else:
                valid = (
                    pa.types.is_string(dtype)
                    or pa.types.is_large_string(dtype)
                    or (
                        pa.types.is_dictionary(dtype)
                        and (
                            pa.types.is_string(dtype.value_type)
                            or pa.types.is_large_string(dtype.value_type)
                        )
                    )
                )
            if not valid:
                raise TypeError(f'Unsupported original type {name}: {dtype}')
        declared = p.metadata.num_rows
        ng = p.num_row_groups
        selected = min(ng, max_groups) if max_groups is not None else ng
        atomic_json(
            root / 'parquet.json',
            dict(
                rows=declared,
                row_groups=ng,
                footer_bytes=p.metadata.serialized_size,
                schema=str(p.schema_arrow),
                selected_groups=selected,
            ),
        )
        offsets = [0]
        for g in range(ng):
            offsets.append(offsets[-1] + p.metadata.row_group(g).num_rows)
        for first in range(0, selected, groups_per_unit):
            last = min(first + groups_per_unit, selected)
            unit = f'{entry["ordinal"]}:{first}:{last}'
            if m.confirmed(unit):
                continue
            if deadline is not None and time.time() >= deadline:
                raise TimeoutError('Approved execution budget reached at unit boundary')
            if shutil.disk_usage(root).free < 100 * 1024**3:
                raise OSError('Disk reserve reached')
            features = []
            relations = []
            quality = Counter()
            types = Counter()
            rows = 0
            missing = 0
            with io_log.open('a') as log:
                log.write(
                    json.dumps(
                        dict(
                            attempt=attempt,
                            operation='reader_requested',
                            groups=list(range(first, last)),
                            mode=mode,
                        )
                    )
                    + '\n'
                )
            if stream_groups:
                prior = len(requests)
                rg_start = time.perf_counter()
                n = 0
                it = p.iter_batches(
                    batch_size=batch_size,
                    row_groups=list(range(first, last)),
                    columns=COLUMNS,
                    use_threads=True,
                )
                while True:
                    t = time.perf_counter()
                    b = next(it, None)
                    times['parquet_read_decode_seconds'] += time.perf_counter() - t
                    if b is None:
                        break
                    t = time.perf_counter()
                    feat, rel, audit = partial(b, entry['ordinal'], offsets[first] + n)
                    times['transform_catalog_group_seconds'] += time.perf_counter() - t
                    for phase, value in audit.get('timings', {}).items():
                        times[phase] += value
                    features.append(feat)
                    relations.append(rel)
                    n += b.num_rows
                    quality.update(audit['quality'])
                    types.update(audit['types'])
                    missing += audit['missing_key']
                    if psutil.Process().memory_info().rss > 10 * 1024**3:
                        raise MemoryError('RSS limit reached')
                if n != offsets[last] - offsets[first]:
                    raise ValueError('Incomplete group range')
                rows = n
                with io_log.open('a') as log:
                    for g in range(first, last):
                        log.write(
                            json.dumps(
                                dict(
                                    attempt=attempt,
                                    group=g,
                                    rows=offsets[g + 1] - offsets[g],
                                    decoder_passes=1,
                                    mode=mode,
                                    shared_reader_first_group=first,
                                    shared_reader_last_group=last,
                                )
                            )
                            + '\n'
                        )
                    log.write(
                        json.dumps(
                            dict(
                                attempt=attempt,
                                operation='shared_reader_ranges',
                                first_group=first,
                                last_group=last,
                                seconds=time.perf_counter() - rg_start,
                                range_fetches=requests[prior:],
                            )
                        )
                        + '\n'
                    )
            else:
                for g in range(first, last):
                    start = offsets[g]
                    n = 0
                    prior = len(requests)
                    rg_start = time.perf_counter()
                    it = p.iter_batches(
                        batch_size=batch_size,
                        row_groups=[g],
                        columns=COLUMNS,
                        use_threads=True,
                    )
                    while True:
                        t = time.perf_counter()
                        b = next(it, None)
                        times['parquet_read_decode_seconds'] += time.perf_counter() - t
                        if b is None:
                            break
                        t = time.perf_counter()
                        feat, rel, audit = partial(b, entry['ordinal'], start + n)
                        times['transform_catalog_group_seconds'] += (
                            time.perf_counter() - t
                        )
                        for phase, value in audit.get('timings', {}).items():
                            times[phase] += value
                        features.append(feat)
                        relations.append(rel)
                        n += b.num_rows
                        quality.update(audit['quality'])
                        types.update(audit['types'])
                        missing += audit['missing_key']
                        if psutil.Process().memory_info().rss > 10 * 1024**3:
                            raise MemoryError('RSS limit reached')
                    expected = p.metadata.row_group(g).num_rows
                    if n != expected:
                        raise ValueError('Incomplete original row group')
                    rows += n
                    with io_log.open('a') as log:
                        log.write(
                            json.dumps(
                                dict(
                                    attempt=attempt,
                                    group=g,
                                    rows=n,
                                    decoder_passes=1,
                                    mode=mode,
                                    seconds=time.perf_counter() - rg_start,
                                    range_fetches=requests[prior:],
                                )
                            )
                            + '\n'
                        )
            t = time.perf_counter()
            feats = pl.concat(features).with_columns(
                pl.col('asset_id').cast(pl.String),
                pl.col('condition_id').cast(pl.String),
            )  # partial keys may repeat; consolidate exactly once downstream
            rels = (
                pl.concat(relations)
                .group_by(['asset_id', 'condition_id'])
                .agg(
                    pl.col('first_ts').min(),
                    pl.col('last_ts').max(),
                    pl.col('events').sum(),
                )
            )
            rels = rels.with_columns(
                pl.col('asset_id').cast(pl.String),
                pl.col('condition_id').cast(pl.String),
            )
            times['partial_merge_seconds'] += time.perf_counter() - t
            if sum(quality.values()) != rows or sum(types.values()) != rows:
                raise ValueError('Audit does not reconcile')
            metadata = dict(
                first_group=first,
                last_group_exclusive=last,
                rows=rows,
                missing_key=missing,
                quality=dict(quality),
                types=dict(types),
                attempt=attempt,
            )
            t = time.perf_counter()
            m.publish(
                unit,
                {'features': feats.to_arrow(), 'relations': rels.to_arrow()},
                metadata,
            )
            times['write_hash_commit_seconds'] += time.perf_counter() - t
            total += rows
            progress = dict(
                source=entry['path'],
                last_group=last,
                total_groups=ng,
                rows_this_attempt=total,
                seconds=time.perf_counter() - started,
                resources=usage(),
                timings=dict(times),
                pid=os.getpid(),
            )
            atomic_json(root / 'progress.json', progress)
            print(json.dumps(progress), flush=True)
    except BaseException as error:
        with (root / 'ingest_attempts.jsonl').open('a') as history:
            history.write(
                json.dumps(
                    dict(
                        attempt=attempt,
                        seconds=time.perf_counter() - started,
                        failed=True,
                    )
                )
                + '\n'
            )
        with io_log.open('a') as log:
            log.write(
                json.dumps(
                    dict(
                        attempt=attempt,
                        operation='attempt_failed',
                        error=repr(error),
                        range_payload_bytes=sum(x['bytes'] for x in requests),
                    )
                )
                + '\n'
            )
        raise
    finally:
        f.close()
        m.db.close()
    attempt_seconds = time.perf_counter() - started
    journal = root / 'ingest_attempts.jsonl'
    if not journal.exists() and (root / 'ingest_report.json').exists():
        previous = json.loads((root / 'ingest_report.json').read_text())
        with journal.open('a') as history:
            history.write(
                json.dumps(
                    dict(
                        attempt='legacy_recorded_attempts', seconds=previous['seconds']
                    )
                )
                + '\n'
            )
    with journal.open('a') as history:
        history.write(
            json.dumps(
                dict(attempt=attempt, seconds=attempt_seconds, failed=False, rows=total)
            )
            + '\n'
        )
    cumulative_seconds = sum(
        json.loads(line)['seconds'] for line in journal.read_text().splitlines()
    )
    m = Manifest(root / 'parts')
    commits = [json.loads(x[0]) for x in m.db.execute('SELECT metadata FROM commits')]
    counted_rows = sum(x['rows'] for x in commits)
    complete = selected == ng and counted_rows == declared
    report = dict(
        source=entry['path'],
        revision=REVISION,
        mode=mode,
        complete=complete,
        declared_rows=declared,
        rows_confirmed=counted_rows,
        row_groups=ng,
        groups_selected=selected,
        seconds=cumulative_seconds,
        current_attempt_seconds=attempt_seconds,
        rows_this_attempt=total,
        events_per_second=total / max(1e-9, time.perf_counter() - started),
        timings=dict(times),
        range_payload_bytes=sum(x['bytes'] for x in requests),
        range_requests=len(requests),
        resources_before=before,
        resources_after=usage(),
        engine_code_sha256=hashlib.sha256(ENGINE_SOURCE.encode()).hexdigest(),
        derived_bytes=sum(x.stat().st_size for x in (root / 'parts').glob('*.parquet')),
        dictionary_read=dictionary_read,
        logical_source_passes=int(total > 0),
        metadata_opens=1,
        stream_groups=stream_groups,
        batch_size=batch_size,
        orphan_objects=m.orphans(),
    )
    atomic_json(root / 'ingest_report.json', report)
    m.db.close()
    return report


# Consolidación de series


def _consolidate(root, buckets=None, deadline=None, sparse_merge=False):
    import duckdb
    import math
    from .storage import digest

    root = Path(root)
    (root / 'engine_consolidation_snapshot.py').write_text(ENGINE_SOURCE)
    m = Manifest(root / 'parts')
    entries = m.db.execute(
        "SELECT path,rows FROM objects WHERE path LIKE '%_features.parquet'"
    ).fetchall()
    parts = [str(root / 'parts' / x[0]) for x in entries]
    expected = sum(
        json.loads(x[0])['rows'] - json.loads(x[0])['missing_key']
        for x in m.db.execute('SELECT metadata FROM commits')
    )
    if not parts:
        raise ValueError('No confirmed features')
    marker = root / 'consolidated.json'
    if marker.exists():
        saved = json.loads(marker.read_text())
        verify_series(saved)
        m.db.close()
        return saved
    buckets = buckets or max(
        8, 2 ** math.ceil(math.log2(max(1, sum(x[1] for x in entries) / 1000000)))
    )
    soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(
        resource.RLIMIT_NOFILE, (min(hard, max(soft, int(buckets) * 3 + 256)), hard)
    )
    db = duckdb.connect()
    db.execute("SET memory_limit='3GB'")
    db.execute('SET threads=4')
    db.execute('SET preserve_insertion_order=false')
    spill = root / 'spill_partitioned'
    spill.mkdir(exist_ok=True)
    db.execute('SET temp_directory=?', [str(spill)])
    db.from_parquet(parts).create_view('parts')
    columns = [x[0] for x in db.execute('DESCRIBE parts').fetchall()]
    expr = []
    for edge, func in [('first', 'arg_min'), ('last', 'arg_max')]:
        names = [c for c in columns if c.endswith('_' + edge)]
        fields = ','.join('"' + n + '" := "' + n + '"' for n in names)
        order = ','.join(
            c + '_' + edge for c in ['event_ts_ms', 'file_ordinal', 'row_position']
        )
        expr.append(
            f'{func}(struct_pack({fields}), ({order})) FILTER (WHERE event_ts_ms_{edge} IS NOT NULL) AS _{edge}'
        )
    for name in columns:
        q = '"' + name + '"'
        if name in KEYS or name.endswith(('_first', '_last')):
            continue
        if name.endswith('_min'):
            expr.append(f'min({q}) AS {q}')
        elif name.endswith('_max'):
            expr.append(f'max({q}) AS {q}')
        elif name == 'reported_trade_size_sum':
            expr.append(f'sum({q}) AS {q}')
        else:
            expr.append(f'sum({q})::UBIGINT AS {q}')
    t = time.perf_counter()
    partition = root / 'merge_partitioned'
    stages = {}
    io_start = usage()
    reused_partition = (root / 'partitioned.json').exists()
    if not (root / 'partitioned.json').exists():
        # Exactly one scan routes every record to one physical bucket. Subsequent bucket
        # queries read disjoint files, never re-filter the full original part list.
        if partition.exists():
            shutil.rmtree(partition)
        # Arrow streaming writes each bucket immediately; no all-partition DuckDB
        # writer buffers or repartition-sort spill. Every source part is scanned once.
        from concurrent.futures import ThreadPoolExecutor

        writers = {}
        buffers = {}
        buffer_rows = {}
        cycle = 0

        def flush_bucket(bucket):
            if not buffer_rows.get(bucket, 0):
                return
            table = pa.concat_tables(buffers[bucket])
            folder = partition / f'bucket={bucket}'
            folder.mkdir(parents=True, exist_ok=True)
            if bucket not in writers:
                writers[bucket] = pq.ParquetWriter(
                    folder / f'data_{cycle:05d}.parquet',
                    table.schema,
                    compression='zstd',
                    compression_level=1,
                )
            writers[bucket].write_table(table, row_group_size=8192)
            buffers[bucket] = []
            buffer_rows[bucket] = 0

        def write_bucket(item):
            key, sub = item
            bucket = int(key[0])
            table = sub.drop('_bucket').to_arrow()
            buffers.setdefault(bucket, []).append(table)
            buffer_rows[bucket] = buffer_rows.get(bucket, 0) + table.num_rows
            if buffer_rows[bucket] >= 8192:
                flush_bucket(bucket)

        def close_cycle():
            for bucket in list(buffers):
                flush_bucket(bucket)
            for writer in writers.values():
                writer.close()
            writers.clear()
            buffers.clear()
            buffer_rows.clear()

        complete = False
        try:
            with ThreadPoolExecutor(max_workers=4) as pool:
                for part_number, part in enumerate(parts):
                    if deadline is not None and time.time() >= deadline:
                        raise TimeoutError(
                            'Approved budget reached during derived partitioning'
                        )
                    for batch in pq.ParquetFile(part).iter_batches(
                        batch_size=1_000_000
                    ):
                        frame = pl.from_arrow(batch).with_columns(
                            (pl.col('asset_id').hash(seed=2026) % int(buckets)).alias(
                                '_bucket'
                            )
                        )
                        list(
                            pool.map(
                                write_bucket,
                                frame.partition_by('_bucket', as_dict=True).items(),
                            )
                        )
                    if (part_number + 1) % 16 == 0:
                        close_cycle()
                        cycle += 1
            close_cycle()
            complete = True
        finally:
            for writer in writers.values():
                writer.close()
            writers.clear()
            buffers.clear()
            buffer_rows.clear()
        import gc

        gc.collect()
        pa.default_memory_pool().release_unused()
        atomic_json(
            root / 'partitioned.json',
            dict(buckets=buckets, input_passes=1, seconds=time.perf_counter() - t),
        )
    if (root / 'partitioned.json').exists():
        buckets = json.loads((root / 'partitioned.json').read_text())['buckets']
    stages['partition_seconds'] = time.perf_counter() - t
    stages['partition_io'] = usage()
    db.execute('SET threads=4')
    aggregate_started = time.perf_counter()
    dest = root / 'series'
    dest.mkdir(exist_ok=True)
    folders = sorted(partition.glob('bucket=*'))
    for bucket_number, folder in enumerate(folders):
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError('Approved budget reached at bucket boundary')
        out = dest / (folder.name + '.parquet')
        if out.exists():
            continue
        db.from_parquet(str(folder / '*.parquet')).create_view(
            'bucket_parts', replace=True
        )
        if sparse_merge:
            db.execute(
                'CREATE OR REPLACE TEMP TABLE duplicate_keys AS SELECT asset_id,condition_id,second FROM bucket_parts GROUP BY asset_id,condition_id,second HAVING count(*)>1'
            ).fetchall()
            projected = ','.join('b."' + c + '"' for c in columns)
            join = 'b.asset_id=d.asset_id AND b.condition_id IS NOT DISTINCT FROM d.condition_id AND b.second=d.second'
            db.execute(
                'CREATE OR REPLACE TEMP TABLE marked AS SELECT '
                + projected
                + ',d.asset_id IS NOT NULL AS _duplicate FROM bucket_parts b LEFT JOIN duplicate_keys d ON '
                + join
            ).fetchall()
            quoted = ','.join('"' + c + '"' for c in columns)
            grouped = (
                'SELECT asset_id,condition_id,second,'
                + ','.join(expr)
                + ' FROM marked WHERE _duplicate GROUP BY asset_id,condition_id,second'
            )
            flattened = (
                'SELECT * EXCLUDE(_first,_last),_first.*,_last.* FROM (' + grouped + ')'
            )
            query = (
                'SELECT '
                + quoted
                + ' FROM marked WHERE NOT _duplicate UNION ALL SELECT '
                + quoted
                + ' FROM ('
                + flattened
                + ') ORDER BY asset_id,condition_id,second'
            )
        else:
            grouped = (
                'SELECT asset_id,condition_id,second,'
                + ','.join(expr)
                + ' FROM bucket_parts GROUP BY asset_id,condition_id,second'
            )
            query = (
                'SELECT * EXCLUDE(_first,_last),_first.*,_last.* FROM ('
                + grouped
                + ') ORDER BY asset_id,condition_id,second'
            )
        tmp = out.with_suffix('.tmp')
        from .columnar import options

        reader = db.sql(query).to_arrow_reader(batch_size=65536)
        writer = None
        try:
            for batch in reader:
                table = pa.Table.from_batches([batch])
                if writer is None:
                    writer = pq.ParquetWriter(
                        tmp, table.schema, **options(table.schema)
                    )
                writer.write_table(table, row_group_size=65536)
        finally:
            reader.close()
            if writer is not None:
                writer.close()
        with tmp.open('rb') as f:
            os.fsync(f.fileno())
        os.replace(tmp, out)
        atomic_json(
            root / 'consolidation_progress.json',
            dict(
                buckets_completed=bucket_number + 1,
                buckets_total=len(folders),
                seconds_current_attempt=time.perf_counter() - t,
                resources=usage(),
            ),
        )
    stages['aggregate_seconds'] = time.perf_counter() - aggregate_started
    stages['aggregate_io'] = usage()
    files = sorted(str(p) for p in dest.glob('*.parquet'))
    db.from_parquet(files).create_view('result')
    check = db.execute(
        'SELECT sum(events),count(*),count(DISTINCT asset_id),count(DISTINCT condition_id) FROM result'
    ).fetchall()[0]
    if int(check[0] or 0) != expected:
        raise ValueError(
            f'Feature event reconciliation failed: {check[0]} != {expected}'
        )
    r = dict(
        seconds=time.perf_counter()
        - t
        + (
            json.loads((root / 'partitioned.json').read_text())['seconds']
            if reused_partition
            else 0
        ),
        current_attempt_seconds=time.perf_counter() - t,
        reused_partition=reused_partition,
        events=int(check[0]),
        series_rows=check[1],
        assets=check[2],
        conditions=check[3],
        sparse_merge=sparse_merge,
        key_only_scan_passes=int(sparse_merge),
        materialized_memory_passes=2 if sparse_merge else 0,
        stage_timings=stages,
        resources_before=io_start,
        input_files=len(parts),
        derived_scan_passes=2,
        validation_output_scan_passes=1,
        output_bytes=sum(Path(p).stat().st_size for p in files),
        outputs=files,
        buckets=buckets,
        integrity_output_scan_passes=1,
        sha256={Path(p).name: digest(p) for p in files},
        resources=usage(),
    )
    atomic_json(marker, r)
    db.close()
    m.db.close()
    return r


def consolidate(root, buckets=None, deadline=None, sparse_merge=False):
    import fcntl

    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    with (root / 'consolidate.lock').open('a') as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        return _consolidate(root, buckets, deadline, sparse_merge)


def process(entry, source, root, **kwargs):
    import fcntl

    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    with (root / 'ingest.lock').open('a') as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        return _process(entry, source, root, **kwargs)


def verify_series(report):
    """Validate canonical objects before reusing a completed partition."""
    from .storage import digest

    rows = 0
    for name in report['outputs']:
        path = Path(name)
        if digest(path) != report['sha256'][path.name]:
            raise ValueError(f'Canonical series checksum mismatch: {path}')
        rows += pq.ParquetFile(path).metadata.num_rows
    if rows != report['series_rows']:
        raise ValueError('Canonical row count mismatch')
    return rows

**src/polymarket_mvp/features.py**

In [ ]:
%%modulo src/polymarket_mvp/features.py
"""Mergeable observable features. This module has no MOMENT target policy."""

import json
import math
import time
import polars as pl
import pyarrow as pa

COLUMNS = [
    'asset_id',
    'condition_id',
    'event_ts_ms',
    'event_type',
    'best_bid',
    'best_ask',
    'bids_json',
    'asks_json',
    'source_archive',
    'size',
    'price',
    'side',
]
STATES = [
    'bilateral',
    'empty_bid',
    'empty_ask',
    'empty_both',
    'boundary',
    'null_quote',
    'invalid_quote',
    'unknown',
]
KEYS = ['asset_id', 'condition_id', 'second']
EDGE = [
    'bid',
    'ask',
    'spread',
    'mid',
    'quality',
    'event_ts_ms',
    'file_ordinal',
    'row_position',
]


def snapshot(bids, asks):
    try:
        parsed = []
        for text, bid in [(bids, True), (asks, False)]:
            if text is None:
                return None, None, 'unknown'
            levels = json.loads(text)
            if not isinstance(levels, list):
                raise ValueError()
            prices = set()
            for level in levels:
                if not isinstance(level, list) or len(level) != 2:
                    raise ValueError()
                p, s = map(float, level)
                if (
                    not math.isfinite(p)
                    or not math.isfinite(s)
                    or not 0 <= p <= 1
                    or s <= 0
                    or p in prices
                ):
                    raise ValueError()
                prices.add(p)
            parsed.append((max(prices) if bid else min(prices)) if prices else None)
        b, a = parsed
        state = (
            'empty_both'
            if b is None and a is None
            else 'empty_bid' if b is None else 'empty_ask' if a is None else None
        )
        return b, a, state
    except (ValueError, TypeError, OverflowError):
        return None, None, 'invalid_quote'


def partial(batch, ordinal, row_start):
    began = time.perf_counter()
    d = pl.from_arrow(batch).with_columns(
        pl.int_range(row_start, row_start + batch.num_rows, dtype=pl.Int64).alias(
            'row_position'
        ),
        pl.lit(ordinal, dtype=pl.Int32).alias('file_ordinal'),
    )
    # JSON is parsed only for snapshots; the dominant event path stays vectorized.
    books = d.filter(pl.col('event_type') == 'book').select(
        'row_position', 'bids_json', 'asks_json'
    )
    overrides = [(i, *snapshot(b, a)) for i, b, a in books.iter_rows()]
    if overrides:
        o = pl.DataFrame(
            overrides,
            schema={
                'row_position': pl.Int64,
                'snapshot_bid': pl.Float64,
                'snapshot_ask': pl.Float64,
                'snapshot_state': pl.String,
            },
            orient='row',
        )
        d = d.join(o, on='row_position', how='left')
    else:
        d = d.with_columns(
            pl.lit(None, dtype=pl.Float64).alias('snapshot_bid'),
            pl.lit(None, dtype=pl.Float64).alias('snapshot_ask'),
            pl.lit(None, dtype=pl.String).alias('snapshot_state'),
        )
    d = d.with_columns(
        pl.when(pl.col('event_type') == 'book')
        .then(pl.col('snapshot_bid'))
        .otherwise(pl.col('best_bid'))
        .alias('bid'),
        pl.when(pl.col('event_type') == 'book')
        .then(pl.col('snapshot_ask'))
        .otherwise(pl.col('best_ask'))
        .alias('ask'),
        pl.col('event_type')
        .is_in(['book', 'price_change'])
        .fill_null(False)
        .alias('is_quote'),
        (pl.col('event_ts_ms') // 1000).alias('second'),
    )
    valid = (
        pl.col('bid').is_finite()
        & pl.col('ask').is_finite()
        & (pl.col('bid') > 0)
        & (pl.col('ask') < 1)
        & (pl.col('bid') <= pl.col('ask'))
    ).fill_null(False)
    d = d.with_columns(
        pl.when(~pl.col('is_quote'))
        .then(pl.lit('unknown'))
        .when(pl.col('snapshot_state').is_not_null())
        .then(pl.col('snapshot_state'))
        .when(valid)
        .then(pl.lit('bilateral'))
        .when(pl.col('bid').is_null() | pl.col('ask').is_null())
        .then(pl.lit('null_quote'))
        .when(
            (~pl.col('bid').is_finite())
            | (~pl.col('ask').is_finite())
            | (pl.col('bid') < 0)
            | (pl.col('ask') > 1)
            | (pl.col('bid') > pl.col('ask'))
        )
        .then(pl.lit('invalid_quote'))
        .when((pl.col('bid') == 0) | (pl.col('ask') == 1))
        .then(pl.lit('boundary'))
        .otherwise(pl.lit('invalid_quote'))
        .alias('quality')
    )
    d = d.with_columns(
        pl.when(pl.col('quality') == 'bilateral')
        .then(pl.col('ask') - pl.col('bid'))
        .alias('spread'),
        pl.when(pl.col('quality') == 'bilateral')
        .then((pl.col('ask') + pl.col('bid')) / 2)
        .alias('mid'),
    )
    keyed = (
        pl.col('asset_id').is_not_null()
        & pl.col('event_ts_ms').is_not_null()
        & (pl.col('event_ts_ms') >= 0)
    )
    audit = {
        'rows': batch.num_rows,
        'missing_key': d.filter(~keyed.fill_null(False)).height,
        'quality': dict(d.group_by('quality').len().iter_rows()),
        'types': dict(
            d.with_columns(pl.col('event_type').fill_null('<null>'))
            .group_by('event_type')
            .len()
            .iter_rows()
        ),
    }
    quality_finished = time.perf_counter()
    relations = (
        d.filter(
            pl.col('asset_id').is_not_null() & pl.col('condition_id').is_not_null()
        )
        .group_by(['asset_id', 'condition_id'])
        .agg(
            pl.col('event_ts_ms').min().alias('first_ts'),
            pl.col('event_ts_ms').max().alias('last_ts'),
            pl.len().cast(pl.UInt64).alias('events'),
        )
    )
    catalog_finished = time.perf_counter()
    d = d.filter(keyed).sort(['event_ts_ms', 'file_ordinal', 'row_position'])
    expr = [
        pl.len().cast(pl.UInt64).alias('events'),
        pl.col('is_quote').sum().cast(pl.UInt64).alias('quotes'),
    ]
    for state in STATES:
        expr.append(
            ((pl.col('quality') == state) & pl.col('is_quote'))
            .sum()
            .cast(pl.UInt64)
            .alias('n_' + state)
        )
    for typ in ['price_change', 'book', 'last_trade_price', 'tick_size_change']:
        expr.append(
            (pl.col('event_type') == typ).sum().cast(pl.UInt64).alias('n_' + typ)
        )
    for source in ['v1', 'v2']:
        expr.append(
            (pl.col('source_archive') == source)
            .sum()
            .cast(pl.UInt64)
            .alias('n_' + source)
        )
    trade = (
        (pl.col('event_type') == 'last_trade_price')
        & pl.col('size').is_finite()
        & (pl.col('size') >= 0)
    )
    expr.append(pl.col('size').filter(trade).sum().alias('reported_trade_size_sum'))
    for side in ['BUY', 'SELL']:
        expr.append(
            ((pl.col('event_type') == 'price_change') & (pl.col('side') == side))
            .sum()
            .cast(pl.UInt64)
            .alias('updates_' + side.lower())
        )
    for col in EDGE:
        for edge in ['first', 'last']:
            v = pl.col(col).filter(pl.col('is_quote'))
            expr.append(getattr(v, edge)().alias(col + '_' + edge))
    for col in ['spread', 'mid', 'price']:
        expr.extend(
            [
                pl.col(col).min().alias(col + '_min'),
                pl.col(col).max().alias(col + '_max'),
            ]
        )
    result = d.group_by(KEYS).agg(expr)
    audit['timings'] = {
        'quality_json_seconds': quality_finished - began,
        'catalog_seconds': catalog_finished - quality_finished,
        'group_seconds': time.perf_counter() - catalog_finished,
    }
    return result, relations, audit


def merge(frame):
    if not frame.height:
        return frame
    expr = []
    for edge in ['first', 'last']:
        names = [c for c in frame.columns if c.endswith('_' + edge)]
        valid = pl.col('event_ts_ms_' + edge).is_not_null()
        order = [
            pl.col(c + '_' + edge).filter(valid)
            for c in ['event_ts_ms', 'file_ordinal', 'row_position']
        ]
        values = pl.struct(names).filter(valid).sort_by(order)
        expr.append(getattr(values, edge)().alias('_' + edge))
    for name in frame.columns:
        if name in KEYS or name.endswith(('_first', '_last')):
            continue
        c = pl.col(name)
        if name.endswith('_min'):
            expr.append(c.min().alias(name))
        elif name.endswith('_max'):
            expr.append(c.max().alias(name))
        else:
            expr.append(c.sum().alias(name))
    return frame.group_by(KEYS).agg(expr).unnest('_first', '_last')

**src/polymarket_mvp/full.py**

In [ ]:
%%modulo src/polymarket_mvp/full.py
"""Full execution behind measured performance and storage gates."""

import argparse
import fcntl
from contextlib import contextmanager
import json
import os
import shutil
import time
from pathlib import Path
from .download import inventory, atomic_json
from .engine import verify_series
from .storage import Manifest
from .windows import build_census, select_training, pack_windows
from .model import train, evaluate, verify_adapter


def verify_full(root):
    root = Path(root)
    inv = inventory(root)
    coverage = []
    enriched = root / 'inventory_enriched.json'
    expected = (
        {x['path']: x for x in json.loads(enriched.read_text())['files']}
        if enriched.exists()
        else {}
    )
    for entry in inv['files']:
        day = entry['path'].split('date=')[1][:10]
        out = root / 'full' / day
        p = out / 'ingest_report.json'
        if not p.exists():
            continue
        report = json.loads(p.read_text())
        identity = json.loads((out / 'source.json').read_text())
        from . import REVISION

        if identity['revision'] != REVISION or identity['source'] != entry:
            raise ValueError('Coverage source identity mismatch')
        if entry['path'] in expected:
            row = expected[entry['path']]
            if (
                report['declared_rows'] != row['rows']
                or report['row_groups'] != row['row_groups']
            ):
                raise ValueError('Original footer count mismatch')
        m = Manifest(out / 'parts')
        m.verify()
        commits = [
            json.loads(r[0]) for r in m.db.execute('SELECT metadata FROM commits')
        ]
        rows = sum(c['rows'] for c in commits)
        groups = sum(c['last_group_exclusive'] - c['first_group'] for c in commits)
        if (
            report['complete']
            and rows == report['declared_rows']
            and groups == report['row_groups']
        ):
            coverage.append(
                dict(
                    path=entry['path'],
                    bytes=entry['bytes'],
                    rows=rows,
                    row_groups=groups,
                    invalid_or_missing_key=sum(c['missing_key'] for c in commits),
                )
            )
        m.db.close()
    r = dict(
        files_confirmed=len(coverage),
        files_expected=164,
        bytes_confirmed=sum(x['bytes'] for x in coverage),
        rows=sum(x['rows'] for x in coverage),
        coverage=coverage,
        data_complete=len(coverage) == 164,
    )
    atomic_json(root / 'coverage.json', r)
    return r


class Mission:
    def __init__(self, root, plan, hours, extra_hours):
        self.root = root
        self.path = root / "mission.json"
        now = time.time()
        if self.path.exists():
            self.data = json.loads(self.path.read_text())
            if self.data["budget_hours"] != hours:
                raise ValueError(
                    "Use the original budget and --extra-hours to extend a mission"
                )
            previous = self.data["attempts"][-1]
            if "ended_utc" not in previous:
                # Last durable heartbeat is a lower bound on activity before a hard interruption.
                previous.update(ended_utc=previous["heartbeat_utc"], interrupted=True)
            if extra_hours:
                self.data["extensions"].append(dict(hours=extra_hours, utc=now))
        else:
            if extra_hours:
                raise ValueError("--extra-hours requires an existing mission")
            self.data = dict(
                started_utc=now,
                budget_hours=hours,
                extensions=[],
                attempts=[],
                revision=plan["revision"],
                benchmark=plan["benchmark_audit"],
                preparation=plan["preflight_receipt"],
            )
        attempts = self.data["attempts"]
        used = sum(a["ended_utc"] - a["started_utc"] for a in attempts)
        self.pause_seconds = sum(
            max(0, b["started_utc"] - a["ended_utc"])
            for a, b in zip(attempts, attempts[1:])
        )
        if attempts:
            self.pause_seconds += max(0, now - attempts[-1]["ended_utc"])
        self.deadline = (
            now
            + 3600 * (hours + sum(x["hours"] for x in self.data["extensions"]))
            - used
        )
        self.attempt = dict(
            started_utc=now, heartbeat_utc=now, pid=os.getpid(), stages=[]
        )
        attempts.append(self.attempt)
        self.save()

    def save(self):
        self.attempt["heartbeat_utc"] = time.time()
        self.data["paused_seconds"] = self.pause_seconds
        atomic_json(self.path, self.data)

    @contextmanager
    def stage(self, name):
        if time.time() >= self.deadline:
            raise TimeoutError(f"Execution budget exhausted before {name}")
        stage = dict(name=name, started_utc=time.time())
        self.attempt["stages"].append(stage)
        self.save()
        atomic_json(
            self.root / "full_status.json",
            dict(
                state=name,
                pid=os.getpid(),
                started_utc=self.data["started_utc"],
                deadline=self.deadline,
            ),
        )
        try:
            yield
            stage["complete"] = True
        finally:
            stage["ended_utc"] = time.time()
            stage["seconds"] = stage["ended_utc"] - stage["started_utc"]
            self.save()

    def heartbeat(self):
        self.save()


def check_inventory(root, entries):
    meta = json.loads((root / "inventory_enriched.json").read_text())
    from . import REVISION

    if meta["revision"] != REVISION or meta["file_count"] != 164:
        raise ValueError("Missing complete pinned footer inventory")
    fields = ("path", "bytes", "sha256", "ordinal", "oid")
    normalized = [{k: e[k] for k in fields} for e in meta["files"]]
    if (
        normalized != entries
        or sum(e["rows"] for e in meta["files"]) != meta["declared_rows"]
    ):
        raise ValueError("Pinned inventory and Parquet footers disagree")


def evaluate_split(root, split, model, mission, reserve_bytes, calibration=None):
    from .storage import digest

    output = root / "full_evaluation" / split
    pack = root / "full_packs" / split
    receipt = output / "verified.json"
    model_identity = digest(
        root / "full_model" / "adapter" / "adapter_model.safetensors"
    )
    index_identity = digest(root / "full_census" / "index.parquet")
    if receipt.exists():
        saved = json.loads(receipt.read_text())
        if (
            saved["adapter_sha256"] != model_identity
            or saved["index_sha256"] != index_identity
        ):
            raise ValueError("Evaluation inputs changed")
        if any(digest(output / name) != sha for name, sha in saved["outputs"].items()):
            raise ValueError(f"Corrupt completed {split} evaluation")
    else:
        with mission.stage(f"packing_{split}"):
            pack_windows(
                root / "full_census",
                pack,
                split,
                deadline=mission.deadline,
                reserve_bytes=reserve_bytes,
            )
        with mission.stage(f"evaluation_{split}"):
            result = evaluate(
                pack, model, output, calibration=calibration, deadline=mission.deadline
            )
            files = [
                "errors.parquet",
                "asset_id.parquet",
                "condition_id.parquet",
                "episodes.parquet",
                "evaluation.json",
            ]
            hashes = {}
            for name in files:
                with (output / name).open("rb") as stream:
                    os.fsync(stream.fileno())
                hashes[name] = digest(output / name)
            saved = dict(
                result=result,
                adapter_sha256=model_identity,
                index_sha256=index_identity,
                pack=json.loads((pack / "pack.json").read_text()),
                outputs=hashes,
            )
            atomic_json(receipt, saved)
    if pack.exists():
        shutil.rmtree(pack)
    return saved["result"]


def execute(root, projection_path, budget_hours=120, extra_hours=0):
    import math
    from . import REVISION

    root = Path(root).resolve()
    if (
        not math.isfinite(budget_hours)
        or not math.isfinite(extra_hours)
        or budget_hours <= 0
        or extra_hours < 0
    ):
        raise ValueError(
            "Execution budgets must be finite, with a positive initial budget"
        )
    root.mkdir(parents=True, exist_ok=True)
    with (root / "full.lock").open("a") as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        plan = json.loads(Path(projection_path).read_text())
        if plan.get("revision") != REVISION or plan.get("full_file_benchmarks") != 4:
            raise ValueError(
                "FULL requires a current pinned projection using four complete benchmarks"
            )
        existing = (root / "mission.json").exists()
        if not existing and (
            not plan["performance_gate"]
            or plan["projected_hours"] > budget_hours
            or not plan["storage_gate"]
            or plan["estimated_storage_requirement"] > shutil.disk_usage(root).free
        ):
            raise RuntimeError(
                "FULL time or storage projection exceeds the supplied budget"
            )
        entries = inventory(root)["files"]
        check_inventory(root, entries)
        import torch

        if not torch.cuda.is_available():
            raise RuntimeError("The projected MOMENT throughput requires CUDA")
        if (root / "full_result.json").exists():
            return json.loads((root / "full_result.json").read_text())
        mission = Mission(root, plan, budget_hours, extra_hours)
        try:
            return run_full(root, entries, plan, mission)
        except BaseException as error:
            atomic_json(
                root / "full_status.json",
                dict(
                    state=(
                        "budget_stopped"
                        if isinstance(error, TimeoutError)
                        else "failed"
                    ),
                    pid=os.getpid(),
                    error=repr(error),
                ),
            )
            raise
        finally:
            mission.attempt["ended_utc"] = time.time()
            mission.save()


def run_full(root, entries, plan, mission):
    from .parallel import run_partitions
    from .catalog import build_catalog

    with mission.stage("download_ingestion_consolidation"):
        pool = run_partitions(
            entries,
            root,
            tag="full",
            workers=plan["workers"],
            consolidators=plan["consolidators"],
            deadline=mission.deadline,
            output_group="full",
            reclaim=True,
            reuse=True,
            disk_reserve=plan["disk_reserve_bytes"],
            scratch_factor=plan["scratch_factor"],
            optimized=True,
            heartbeat=mission.heartbeat,
        )
        if not pool["complete"]:
            if time.time() >= mission.deadline:
                raise TimeoutError(
                    "Execution budget reached; committed partitions retained"
                )
            raise RuntimeError(
                f'Incomplete originals; committed results retained: {pool["errors"]}'
            )
    # Comprobar la cobertura antes de preparar el modelo.
    with mission.stage("coverage_verification"):
        coverage = verify_full(root)
        if not coverage["data_complete"]:
            raise RuntimeError("Full input coverage not verified")
        partition_roots = [
            root / "full" / e["path"].split("date=")[1][:10] for e in entries
        ]
        series = []
        for part in partition_roots:
            r = json.loads((part / "consolidated.json").read_text())
            series.extend(r["outputs"])
    with mission.stage("catalog"):
        catalog = build_catalog(partition_roots, root / "catalog")
    with mission.stage("census"):
        certificate = root / "canonical_day_certificate.json"
        from .storage import digest

        proof = json.loads(certificate.read_text()) if certificate.exists() else {}
        canonical_unique = proof.get(
            "disjoint_utc_day_timestamp_ranges", False
        ) and proof.get("inventory_sha256") == digest(root / "inventory_enriched.json")
        census = build_census(
            series,
            root / "full_census",
            deadline=mission.deadline,
            canonical_unique=canonical_unique,
        )
    counts = census["split_counts"]
    eval_seconds = (counts["validation"] + counts["test"]) / plan[
        "evaluation_windows_per_second"
    ]
    packing_seconds = (
        plan["packing_seconds"]
        * census["eligible"]
        / max(1, plan["estimated_eligible_windows"])
    )
    budget = max(
        0,
        int(
            (mission.deadline - time.time() - eval_seconds - packing_seconds)
            * plan["training_windows_per_second"]
            / 1.2
        ),
    )
    free = shutil.disk_usage(root).free
    existing_train_bytes = sum(
        p.stat().st_size for p in (root / "full_packs/train").glob("*.npy")
    )
    evaluation_reserve = (
        int(1.2 * max(counts["validation"], counts["test"]) * 2562) + 2 * 1024**3
    )
    disk_cap = max(
        0,
        (free + existing_train_bytes - plan["disk_reserve_bytes"] - evaluation_reserve)
        // 3100,
    )
    budget = min(budget, disk_cap, counts["train"])
    with mission.stage("selection"):
        selection = select_training(root / "full_census", budget)
        if selection["selected"] > disk_cap:
            raise OSError(
                f"Frozen selection needs {selection['selected']} windows; disk cap is {disk_cap}"
            )
        tensor_bytes = dict(
            train=selection["selected"] * 2562,
            validation=counts["validation"] * 2562,
            test=counts["test"] * 2562,
        )
        atomic_json(
            root / "full_pack_budget.json",
            dict(
                windows=counts,
                selected=selection["selected"],
                tensor_bytes=tensor_bytes,
                disk_cap_windows=disk_cap,
                policy="train plus one evaluation split at a time",
                disk_available_bytes=shutil.disk_usage(root).free,
                reserve_bytes=plan["disk_reserve_bytes"],
            ),
        )
    with mission.stage("packing_train"):
        pack_windows(
            root / "full_census",
            root / "full_packs/train",
            "train",
            selected=True,
            deadline=mission.deadline,
            reserve_bytes=plan["disk_reserve_bytes"],
        )
    with mission.stage("training"):
        model, trained = train(
            root / "full_packs/train",
            root / "full_model",
            microbatch=plan["microbatch"],
            resume=(root / "full_model/checkpoint.json").exists(),
            deadline=mission.deadline,
        )
        if not trained["complete"]:
            raise TimeoutError("Training budget reached; checkpoint retained")
        adapter_check = verify_adapter(root / "full_model")
    validation = evaluate_split(
        root, "validation", model, mission, plan["disk_reserve_bytes"]
    )
    test = evaluate_split(
        root,
        "test",
        model,
        mission,
        plan["disk_reserve_bytes"],
        calibration=validation["calibration"],
    )
    result = dict(
        complete=True,
        coverage=coverage,
        catalog=catalog,
        census=census,
        selection=selection,
        training=trained,
        adapter_verification=adapter_check,
        validation=validation,
        test=test,
        elapsed_seconds=time.time() - mission.data["started_utc"],
    )
    atomic_json(root / "full_result.json", result)
    atomic_json(
        root / "full_status.json",
        dict(state="complete", pid=os.getpid(), finished_utc=time.time()),
    )
    return result


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="runs/mvp")
    p.add_argument("--projection", required=True)
    p.add_argument("--budget-hours", type=float, default=120)
    p.add_argument("--extra-hours", type=float, default=0)
    args = p.parse_args()
    execute(args.root, args.projection, args.budget_hours, args.extra_hours)

**src/polymarket_mvp/gates.py**

In [ ]:
%%modulo src/polymarket_mvp/gates.py
"""Explicit feasibility gates; estimates are never completion evidence."""

from dataclasses import dataclass, asdict


@dataclass(frozen=True)
class Projection:
    processing_seconds: float
    census_seconds: float
    training_seconds: float
    evaluation_seconds: float
    elapsed_seconds: float
    minimum_windows: int
    budget_windows: int
    full_file_benchmarks: int

    def report(self):
        result = asdict(self)
        total = sum(
            (
                self.processing_seconds,
                self.census_seconds,
                self.training_seconds,
                self.evaluation_seconds,
                self.elapsed_seconds,
            )
        )
        result.update(
            projected_hours=total / 3600,
            performance_gate=self.full_file_benchmarks >= 3 and total <= 48 * 3600,
            cardinality_gate=self.minimum_windows <= self.budget_windows,
        )
        return result

    def require_full(self):
        r = self.report()
        if not r['performance_gate']:
            raise RuntimeError(f'FULL blocked pending measured alternatives: {r}')
        return r

**src/polymarket_mvp/io_audit.py**

In [ ]:
%%modulo src/polymarket_mvp/io_audit.py
"""Summarize actual completed decoder passes, keeping unknown measurements explicit."""

import json
from collections import Counter
from pathlib import Path
from .download import atomic_json


def audit(root):
    root = Path(root)
    completed = Counter()
    requested = Counter()
    runs = []
    missing = []
    for folder in sorted((root / 'benchmarks').iterdir()):
        identity = folder / 'source.json'
        log = folder / 'original_reads.jsonl'
        if not identity.exists() or not log.exists():
            continue
        source = json.loads(identity.read_text())['source']['path']
        local = Counter()
        seen_attempts = set()
        for line in log.open():
            r = json.loads(line)
            if r.get('operation') == 'reader_requested':
                for group in r['groups']:
                    requested[(source, group)] += 1
            elif 'group' in r and 'decoder_passes' in r:
                local[r['group']] += r['decoder_passes']
                seen_attempts.add(r.get('attempt'))
        for group, count in local.items():
            completed[(source, group)] += count
        runs.append(
            dict(
                run=folder.name,
                source=source,
                groups_seen=len(local),
                completed_decoder_visits=sum(local.values()),
                max_passes_per_group=max(local.values(), default=0),
                attempts=len(seen_attempts),
            )
        )
    # Cloned consolidation benchmarks may carry a copied read log for lineage; do
    # not sum these as fresh physical input work. Their source reference is retained.
    unique = []
    for r in runs:
        if 'consolidation_' not in r['run'] and not (r['run'] == '2026-02-26_local_v5'):
            unique.append(r)
    downloads = []
    for p in (root / 'staging').glob('*.download.json'):
        downloads.append(json.loads(p.read_text()))
    remote = []
    for p in (root / 'benchmarks').glob('*/benchmark.json'):
        r = json.loads(p.read_text())
        ingest = r.get('ingest', {})
        if ingest.get('mode') == 'remote':
            if ingest.get('range_payload_bytes') is None:
                missing.append(str(p))
            else:
                remote.append(
                    dict(
                        run=p.parent.name,
                        bytes=ingest['range_payload_bytes'],
                        requests=ingest['range_requests'],
                    )
                )
    result = dict(
        runs=unique,
        decoder_visits_logged=sum(r['completed_decoder_visits'] for r in unique),
        original_download_payload_bytes=sum(r['payload_bytes'] for r in downloads),
        remote_measured=remote,
        unknown_legacy_transfer_measurements=missing,
        caveats=[
            'Original decoder passes are distinct from physical disk reads served by cache',
            'Standalone decode and previous-architecture benchmarks have separate ledgers',
            'HTTP counters measure response payload, excluding protocol overhead',
        ],
    )
    atomic_json(root / 'io_audit.json', result)
    return result

**src/polymarket_mvp/model.py**

In [ ]:
%%modulo src/polymarket_mvp/model.py
"""Real MOMENT LoRA optimization, resume, throughput, and paired evaluation."""

import contextlib
import copy
import json
import math
import os
import random
import time
import uuid
import inspect
import hashlib
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset, DataLoader
from momentfm import MOMENTPipeline
from peft import LoraConfig, get_peft_model
from . import MODEL_REVISION
from .download import atomic_json
from .storage import digest

# Entradas del modelo


class WindowDataset(Dataset):
    def __init__(self, root):
        self.root = Path(root)
        self.x = np.load(self.root / 'x.npy', mmap_mode='r')
        self.observed = np.load(self.root / 'observed.npy', mmap_mode='r')
        self.target = np.load(self.root / 'target.npy', mmap_mode='r')

    def __len__(self):
        return len(self.target)

    def __getitem__(self, i):
        target = int(self.target[i])
        values = np.array(self.x[i], copy=True)
        observed = np.array(self.observed[i], copy=True)
        y = values[target : target + 8].copy()
        values[target:] = 0  # Ocultar el objetivo y los valores posteriores.
        observed[target + 8 :] = 0
        visible = observed.copy()
        visible[target:] = 0
        return dict(
            x=torch.from_numpy(values).unsqueeze(0),
            input_mask=torch.from_numpy(observed).long(),
            mask=torch.from_numpy(visible).long(),
            y=torch.from_numpy(y),
            target=target,
            index=i,
        )


# Modelo y adaptadores


def load_base(device):
    model = MOMENTPipeline.from_pretrained(
        'AutonLab/MOMENT-1-small',
        revision=MODEL_REVISION,
        model_kwargs={'task_name': 'reconstruction'},
    )
    model.init()
    return model.to(device)


def adapter(base):
    return get_peft_model(
        base,
        LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.0,
            target_modules=['q', 'v'],
            modules_to_save=['head'],
            bias='none',
        ),
    )


def prediction(model, batch, device):
    positions = (
        batch['target'].to(device)[:, None] + torch.arange(8, device=device)[None, :]
    )
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16, enabled=device.type == 'cuda'
    ):
        out = model(
            x_enc=batch['x'].to(device),
            input_mask=batch['input_mask'].to(device),
            mask=batch['mask'].to(device),
        ).reconstruction[:, 0]
    return out.gather(1, positions).float()


def sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize()


# Guardado y recuperación


def checkpoint(root, model, optimizer, state):
    root = Path(root)
    name = f'checkpoint_{state["step"]:09d}_{uuid.uuid4().hex}.pt'
    tmp = root / (name + '.tmp')
    payload = dict(
        state,
        trainable={
            n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad
        },
        optimizer=optimizer.state_dict(),
        torch_rng=torch.get_rng_state(),
        numpy_rng=np.random.get_state(),
        python_rng=random.getstate(),
        cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    )
    with tmp.open('wb') as f:
        torch.save(payload, f)
        f.flush()
        os.fsync(f.fileno())
    sha = digest(tmp)
    os.replace(tmp, root / name)
    atomic_json(
        root / 'checkpoint.json', dict(file=name, sha256=sha, step=state['step'])
    )


def train(
    pack,
    root,
    microbatch=32,
    effective_batch=256,
    resume=False,
    max_steps=None,
    benchmark=False,
    deadline=None,
):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    pack_receipt = json.loads((Path(pack) / 'pack.json').read_text())
    if 'hashes' not in pack_receipt:
        raise ValueError('Rebuild legacy pack to establish tensor integrity')
    for filename, sha in pack_receipt['hashes'].items():
        if digest(Path(pack) / filename) != sha:
            raise ValueError(f'Corrupt training pack: {filename}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(2026)
    np.random.seed(2026)
    random.seed(2026)
    data = WindowDataset(pack)
    code_hash = hashlib.sha256(
        '\n'.join(
            inspect.getsource(x) for x in [WindowDataset, adapter, prediction, train]
        ).encode()
    ).hexdigest()
    identity = dict(
        pack_hashes=pack_receipt['hashes'],
        code_sha256=code_hash,
        model_revision=MODEL_REVISION,
        pack_sha256=digest(Path(pack) / 'metadata.parquet'),
        microbatch=microbatch,
        effective_batch=effective_batch,
        lr=1e-4,
        seed=2026,
        windows=len(data),
    )
    model = adapter(load_base(device))
    params = {n: p for n, p in model.named_parameters() if p.requires_grad}
    opt = torch.optim.AdamW(params.values(), lr=1e-4, weight_decay=0.01)
    permutation = np.random.default_rng(2026).permutation(len(data))
    cursor = 0
    step = 0
    history = []
    prior_wall = 0.0
    if resume:
        meta = json.loads((root / 'checkpoint.json').read_text())
        path = root / meta['file']
        if digest(path) != meta['sha256']:
            raise ValueError('Checkpoint integrity failure')
        saved = torch.load(
            path, map_location='cpu', weights_only=False
        )  # only locally created verified checkpoint
        if saved['identity'] != identity:
            raise ValueError('Incompatible checkpoint')
        for n, p in params.items():
            p.data.copy_(saved['trainable'][n].to(device))
        opt.load_state_dict(saved['optimizer'])
        cursor = saved['cursor']
        step = saved['step']
        history = saved['history']
        prior_wall = saved.get(
            'training_wall_seconds', sum(x['seconds'] for x in history)
        )
        torch.set_rng_state(saved['torch_rng'])
        np.random.set_state(saved['numpy_rng'])
        random.setstate(saved['python_rng'])
        if saved['cuda_rng'] and device.type == 'cuda':
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
    elif (root / 'checkpoint.json').exists():
        raise FileExistsError('Checkpoint exists; request resume')
    start = time.perf_counter()
    saved_at = start
    trained_start = cursor
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    # Check both LoRA and head updates; frozen tensors are checked by version counters.
    before = {n: p.detach().clone() for n, p in params.items()}
    frozen = {n: p._version for n, p in model.named_parameters() if not p.requires_grad}
    grad_verified = False
    model.train()
    while cursor < len(data):
        if deadline is not None and time.time() >= deadline:
            break
        if max_steps is not None and step >= max_steps:
            break
        indices = permutation[cursor : min(cursor + effective_batch, len(data))]
        opt.zero_grad(set_to_none=True)
        loss_sum = 0.0
        sync(device)
        t = time.perf_counter()
        for offset in range(0, len(indices), microbatch):
            ids = indices[offset : offset + microbatch]
            batch = torch.utils.data.default_collate([data[int(i)] for i in ids])
            pred = prediction(model, batch, device)
            loss = (pred - batch['y'].to(device)).square().mean()
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite loss')
            (loss * len(ids) / len(indices)).backward()
            loss_sum += float(loss.detach()) * len(ids)
        finite = all(
            p.grad is None or bool(torch.isfinite(p.grad).all())
            for p in params.values()
        )
        lora_grad = sum(
            float(p.grad.float().abs().sum())
            for n, p in params.items()
            if 'lora_' in n and p.grad is not None
        )
        if not finite or not lora_grad > 0:
            raise ValueError('Missing or invalid LoRA gradients')
        grad_verified = True
        torch.nn.utils.clip_grad_norm_(params.values(), 1.0)
        opt.step()
        cursor += len(indices)
        step += 1
        sync(device)
        seconds = time.perf_counter() - t
        rec = dict(
            step=step,
            windows=len(indices),
            mse=loss_sum / len(indices),
            seconds=seconds,
            windows_per_second=len(indices) / seconds,
        )
        history.append(rec)
        with (root / 'history.jsonl').open('a') as f:
            f.write(json.dumps(rec) + '\n')
        if step % 500 == 0 or time.perf_counter() - saved_at >= 300:
            checkpoint(
                root,
                model,
                opt,
                dict(
                    identity=identity,
                    cursor=cursor,
                    step=step,
                    history=history,
                    training_wall_seconds=prior_wall + time.perf_counter() - start,
                ),
            )
            saved_at = time.perf_counter()
    checkpoint(
        root,
        model,
        opt,
        dict(
            identity=identity,
            cursor=cursor,
            step=step,
            history=history,
            training_wall_seconds=prior_wall + time.perf_counter() - start,
        ),
    )
    model.save_pretrained(root / 'adapter', safe_serialization=True)
    changes = {
        n: float((p.detach() - before[n]).abs().max()) for n, p in params.items()
    }
    if cursor > trained_start:
        if not any(v > 0 for n, v in changes.items() if 'lora_' in n):
            raise ValueError('LoRA did not update')
        if not any(v > 0 for n, v in changes.items() if 'head' in n):
            raise ValueError('Head did not update')
    if any(p._version != frozen[n] for n, p in model.named_parameters() if n in frozen):
        raise ValueError('Frozen weights changed')
    measured = history[min(5, max(0, len(history) - 1)) :]
    throughput = sum(x['windows'] for x in measured) / max(
        1e-9, sum(x['seconds'] for x in measured)
    )
    report = dict(
        identity=identity,
        complete=cursor == len(data),
        benchmark=benchmark,
        trained_windows=cursor,
        optimizer_steps=step,
        elapsed_seconds=prior_wall + time.perf_counter() - start,
        current_attempt_seconds=time.perf_counter() - start,
        optimizer_step_seconds=sum(x['seconds'] for x in history),
        windows_per_second=throughput,
        gradient_verified=grad_verified,
        parameter_changes=changes,
        trainable_parameters=sum(p.numel() for p in params.values()),
        vram_peak=torch.cuda.max_memory_allocated() if device.type == 'cuda' else 0,
        epoch_definition='one pass over frozen selection; not over the eligible census',
    )
    atomic_json(root / 'training.json', report)
    return model, report


# Evaluación


def evaluate(pack, trained, root, batch_size=64, calibration=None, deadline=None):
    receipt = json.loads((Path(pack) / 'pack.json').read_text())
    if 'hashes' not in receipt:
        raise ValueError('Rebuild legacy evaluation pack to establish tensor integrity')
    for filename, sha in receipt['hashes'].items():
        if digest(Path(pack) / filename) != sha:
            raise ValueError(f'Corrupt evaluation pack: {filename}')
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    device = next(trained.parameters()).device
    base = load_base(device)
    base.eval()
    trained.eval()
    data = WindowDataset(pack)
    metadata_rows = (
        row
        for batch in pq.ParquetFile(Path(pack) / 'metadata.parquet').iter_batches(
            batch_size=4096
        )
        for row in batch.to_pylist()
    )
    metadata_cursor = 0
    t = time.perf_counter()
    records = []
    writer = None
    output = root / 'errors.parquet'
    with torch.inference_mode():
        for batch in DataLoader(
            data, batch_size=batch_size, shuffle=False, num_workers=0
        ):
            if deadline is not None and time.time() >= deadline:
                if writer:
                    writer.close()
                raise TimeoutError(
                    'Approved budget reached during evaluation; no complete evaluation published'
                )
            original = prediction(base, batch, device).cpu()
            adjusted = prediction(trained, batch, device).cpu()
            # Last visible observation, not padded zero or a hidden target.
            positions = torch.arange(512)[None, :].expand(len(batch['x']), -1)
            last = torch.where(batch['mask'].bool(), positions, -1).max(1).values
            persistence = batch['x'][:, 0].gather(1, last[:, None]).expand(-1, 8)
            result = []
            for i, index in enumerate(batch['index'].tolist()):
                if index != metadata_cursor:
                    raise ValueError('Evaluation metadata order mismatch')
                meta = next(metadata_rows)
                metadata_cursor += 1
                row = {
                    k: meta[k]
                    for k in [
                        'window_id',
                        'asset_id',
                        'condition_id',
                        'start',
                        'target',
                    ]
                }
                row.update(
                    channel='spread',
                    source=meta.get('source', 'not_recorded'),
                    activity=meta.get('activity'),
                    observed=meta.get('observed'),
                    window_file=meta.get('file'),
                )
                for name, pred in [
                    ('persistence', persistence),
                    ('moment', original),
                    ('lora', adjusted),
                ]:
                    err = pred[i] - batch['y'][i]
                    row[name + '_mae'] = float(err.abs().mean())
                    row[name + '_mse'] = float(err.square().mean())
                row.update(
                    observed_spread=float(batch['y'][i].mean()),
                    last_context=float(persistence[i, 0]),
                    direction=float(batch['y'][i].mean() - persistence[i, 0]),
                )
                result.append(row)
            table = pa.Table.from_pylist(result)
            if writer is None:
                writer = pq.ParquetWriter(output, table.schema, compression='zstd')
            writer.write_table(table)
    if writer:
        writer.close()
    del base
    import duckdb

    db = duckdb.connect()
    db.execute("SET memory_limit='2GB'")
    db.execute('SET temp_directory=?', [str(root / 'spill')])
    db.from_parquet(str(output)).create_view('errors')
    aggregate = ','.join(
        f'avg({n}_mae) AS {n}_mae,sqrt(avg({n}_mse)) AS {n}_rmse'
        for n in ['persistence', 'moment', 'lora']
    )
    overall = (
        db.execute('SELECT count(*) AS windows,' + aggregate + ' FROM errors')
        .fetchdf()
        .to_dict('records')[0]
    )
    for key in ['asset_id', 'condition_id']:
        db.execute(
            f"COPY (SELECT {key},count(*) AS windows,{aggregate} FROM errors GROUP BY {key}) TO '{root/(key+'.parquet')}' (FORMAT PARQUET,COMPRESSION ZSTD)"
        ).fetchall()
    macro = (
        db.execute(
            'SELECT '
            + ','.join(
                f'avg({n}_mae) AS macro_mae_{n}'
                for n in ['persistence', 'moment', 'lora']
            )
            + ' FROM read_parquet(?)',
            [str(root / 'condition_id.parquet')],
        )
        .fetchdf()
        .to_dict('records')[0]
    )
    overall.update(macro)
    if calibration is None:
        calibration = float(
            db.execute('SELECT quantile_cont(lora_mae,0.995) FROM errors').fetchall()[
                0
            ][0]
        )
    episodes = root / 'episodes.parquet'
    db.execute(
        f"COPY (SELECT condition_id,start,max(lora_mae) AS score,count(*) AS assets,min(start+target) AS target_start,max(start+target+8) AS target_end FROM errors WHERE lora_mae>={float(calibration)} GROUP BY condition_id,start ORDER BY score DESC) TO '{episodes}' (FORMAT PARQUET,COMPRESSION ZSTD)"
    ).fetchall()
    overall.update(
        seconds=time.perf_counter() - t, calibration=calibration, episodes=str(episodes)
    )
    atomic_json(root / 'evaluation.json', overall)
    db.close()
    return overall


def verify_adapter(root, device=None):
    """Reload the exported PEFT adapter/head and compare every trained tensor."""
    from peft import PeftModel

    root = Path(root)
    meta = json.loads((root / 'checkpoint.json').read_text())
    path = root / meta['file']
    if digest(path) != meta['sha256']:
        raise ValueError('Checkpoint integrity failure')
    saved = torch.load(path, map_location='cpu', weights_only=False)
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    reloaded = PeftModel.from_pretrained(
        load_base(device), root / 'adapter', is_trainable=False
    )
    actual = dict(reloaded.named_parameters())
    difference = 0.0
    for name, tensor in saved['trainable'].items():
        if name not in actual:
            raise ValueError(f'Exported trained tensor missing: {name}')
        difference = max(
            difference, float((actual[name].detach().cpu() - tensor).abs().max())
        )
    if difference != 0:
        raise ValueError(f'Adapter reload differs from checkpoint: {difference}')
    result = dict(
        passed=True,
        trained_tensors=len(saved['trainable']),
        max_parameter_difference=difference,
        model_revision=MODEL_REVISION,
        checkpoint_sha256=meta['sha256'],
    )
    atomic_json(root / 'adapter_verification.json', result)
    return result

**src/polymarket_mvp/parallel.py**

In [ ]:
%%modulo src/polymarket_mvp/parallel.py
"""Measured file-level concurrency with independent manifests and stage admission."""

import json, os, shutil, subprocess, sys, time, threading
from contextlib import contextmanager
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import psutil
from .download import download, atomic_json
from . import REVISION
from .engine import verify_series


def run_partitions(
    entries,
    root,
    tag='parallel4',
    workers=4,
    consolidators=2,
    deadline=None,
    output_group='benchmarks',
    reclaim=False,
    reuse=False,
    disk_reserve=0,
    scratch_factor=8.5,
    ram_budget=20 * 1024**3,
    optimized=False,
    heartbeat=None,
):
    root = Path(root)
    start = time.perf_counter()
    sem = threading.Semaphore(consolidators)
    admission = threading.Lock()
    running = {}
    records = []
    held = dict(disk=0, ram=0)  # Recursos reservados por los trabajos activos.

    @contextmanager
    def reserve(kind, need):
        if not need:
            yield
            return
        while True:
            if deadline and time.time() >= deadline:
                raise TimeoutError(f'Approved budget reached while waiting for {kind}')
            with admission:
                if kind == 'disk':
                    capacity = shutil.disk_usage(root).free - disk_reserve
                    if capacity < need and not held[kind]:
                        raise RuntimeError(
                            f'Insufficient staging space: need {need}, have {capacity} above reserve'
                        )
                else:
                    capacity = (
                        ram_budget if psutil.virtual_memory().available >= need else 0
                    )
                if capacity - held[kind] >= need:
                    held[kind] += need
                    break
            time.sleep(2)
        try:
            yield
        finally:
            with admission:
                held[kind] -= need

    def run_child(args, log_path):
        env = dict(os.environ, POLARS_MAX_THREADS='4', OMP_NUM_THREADS='4')
        with log_path.open('a') as log:
            process = subprocess.Popen(
                [sys.executable, '-u', '-m', 'polymarket_mvp.worker', *args],
                stdout=log,
                stderr=subprocess.STDOUT,
                env=env,
            )
            with admission:
                running[process.pid] = dict(arguments=args, started_utc=time.time())
            code = process.wait()
            with admission:
                running.pop(process.pid, None)
        if code:
            raise RuntimeError(
                f'Worker failed ({code}); committed progress retained: {log_path}'
            )

    def run_one(entry):
        if deadline and time.time() >= deadline:
            raise TimeoutError(
                'Approved budget reached before downloading next original'
            )
        day = entry['path'].split('date=')[1][:10]
        out = (
            root / 'full' / day
            if output_group == 'full'
            else root / 'benchmarks' / f'{day}_local_{tag}'
        )
        if reuse and not out.exists():
            import hashlib

            feature_hash = hashlib.sha256(
                Path(__file__).with_name('features.py').read_bytes()
            ).hexdigest()
            candidates = sorted(
                (root / 'benchmarks').glob(day + '_local_*'),
                key=lambda p: p.stat().st_mtime,
                reverse=True,
            )
            for candidate in candidates:
                if not (candidate / 'benchmark.json').exists():
                    continue
                if not (candidate / 'source.json').exists():
                    continue
                identity = json.loads((candidate / 'source.json').read_text())
                if (
                    identity['source'] == entry
                    and identity['feature_code_sha256'] == feature_hash
                    and identity['revision'] == REVISION
                ):
                    try:
                        candidate_report = json.loads(
                            (candidate / 'benchmark.json').read_text()
                        )
                        if not candidate_report['complete']:
                            continue
                        verify_series(candidate_report['consolidation'])
                    except (ValueError, OSError, KeyError):
                        continue
                    out.parent.mkdir(parents=True, exist_ok=True)
                    out.symlink_to(candidate.resolve(), target_is_directory=True)
                    break
        out.mkdir(parents=True, exist_ok=True)
        done = out / 'benchmark.json'
        if done.exists():
            identity = json.loads((out / 'source.json').read_text())
            if identity['revision'] != REVISION or identity['source'] != entry:
                raise ValueError('Completed partition source identity mismatch')
            r = json.loads(done.read_text())
            verify_series(r['consolidation'])
            return dict(r, reused_completed=True)
        with reserve(
            'disk', int(entry['bytes'] * scratch_factor) if disk_reserve else 0
        ):
            began = time.perf_counter()
            atomic_json(
                out / 'status.json',
                dict(state='waiting_memory', pid=os.getpid(), started_utc=time.time()),
            )
            download_started = time.perf_counter()
            cache_existed = (root / 'staging' / (day + '.parquet')).exists()
            source = root / 'staging' / (day + '.parquet')
            receipt = source.with_suffix('.download.json')
            if (out / 'ingest_report.json').exists() and receipt.exists():
                network = json.loads(receipt.read_text())
            else:
                source, network = download(entry, root / 'staging', deadline=deadline)
            download_elapsed = time.perf_counter() - download_started
            if not (out / 'ingest_report.json').exists():
                atomic_json(
                    out / 'status.json', dict(state='ingestion', pid=os.getpid())
                )
                args = [
                    'ingest',
                    '--out',
                    str(out),
                    '--source',
                    str(source),
                    '--entry',
                    json.dumps(entry),
                    '--groups-per-unit',
                    '16',
                ]
                if deadline:
                    args += ['--deadline', str(deadline)]
                if optimized:
                    args += ['--dictionary-read']
                with reserve('ram', 5 * 1024**3):
                    run_child(args, out / 'ingest.log')
            ingest = json.loads((out / 'ingest_report.json').read_text())
            if not ingest['complete']:
                raise RuntimeError('Original file incomplete')
            with sem, reserve('ram', 6 * 1024**3):
                atomic_json(
                    out / 'status.json', dict(state='consolidation', pid=os.getpid())
                )
                args = ['consolidate', '--out', str(out)]
                if deadline:
                    args += ['--deadline', str(deadline)]
                if optimized:
                    args += ['--sparse-merge']
                run_child(args, out / 'consolidation.log')
            merged = json.loads((out / 'consolidated.json').read_text())
            r = dict(
                complete=True,
                source_bytes=entry['bytes'],
                ingest=ingest,
                download=network,
                consolidation=merged,
                wall_seconds=time.perf_counter() - began,
                network_cache_reused=cache_existed,
                concurrency=dict(ingesters=workers, consolidators=consolidators),
                timing_scope='actual queued job wall; component stage timings separate',
            )
            r['current_download_wall_seconds'] = download_elapsed
            verification_started = time.perf_counter()
            if reclaim:
                from .storage import Manifest

                m = Manifest(out / 'parts')
                m.compact(merged['outputs'], merged, delete_originals=True)
                m.db.close()
                source.unlink(missing_ok=True)
                for name in ['merge_partitioned', 'spill', 'spill_partitioned']:
                    if (out / name).exists():
                        shutil.rmtree(out / name)
            r['verify_and_reclaim_seconds'] = time.perf_counter() - verification_started
            atomic_json(done, r)
        atomic_json(
            out / 'status.json',
            dict(state='complete', finished_utc=time.time(), pid=os.getpid()),
        )
        print('PARTITION_COMPLETE', day, r['wall_seconds'], flush=True)
        return r

    errors = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(run_one, e) for e in entries]
        while futures:
            ready = [f for f in futures if f.done()]
            for f in ready:
                try:
                    records.append(f.result())
                except BaseException as error:
                    errors.append(repr(error))
                futures.remove(f)
            with admission:
                active = dict(running)
                reservations = dict(held)
            atomic_json(
                root / f'{tag}_progress.json',
                dict(
                    pid=os.getpid(),
                    complete_files=len(records),
                    errors=errors,
                    total_files=len(entries),
                    active_workers=active,
                    reservations=reservations,
                    seconds=time.perf_counter() - start,
                    memory_available=psutil.virtual_memory().available,
                ),
            )
            if heartbeat:
                heartbeat()
            if futures:
                time.sleep(2)
    stage_seconds = sum(
        r['ingest']['seconds'] + r['consolidation']['seconds'] for r in records
    )
    report = dict(
        complete=not errors and len(records) == len(entries),
        errors=errors,
        files=len(records),
        rows=sum(r['ingest']['rows_confirmed'] for r in records),
        original_bytes=sum(r['source_bytes'] for r in records),
        wall_seconds=time.perf_counter() - start,
        component_seconds_sum=stage_seconds,
        observed_overlap_factor=stage_seconds / (time.perf_counter() - start),
        reused_partitions=sum(r.get('reused_completed', False) for r in records),
        new_download_worker_seconds=sum(
            r.get('current_download_wall_seconds', 0)
            for r in records
            if not r.get('reused_completed')
        ),
        new_ingest_worker_seconds=sum(
            r['ingest']['seconds'] for r in records if not r.get('reused_completed')
        ),
        new_consolidation_worker_seconds=sum(
            r['consolidation']['seconds']
            for r in records
            if not r.get('reused_completed')
        ),
        new_verify_reclaim_worker_seconds=sum(
            r.get('verify_and_reclaim_seconds', 0)
            for r in records
            if not r.get('reused_completed')
        ),
        workers=workers,
        consolidators=consolidators,
        benchmark_only=output_group == 'benchmarks',
        records=[r['ingest']['source'] for r in records],
    )
    atomic_json(root / f'{tag}_benchmark.json', report)
    return report

**src/polymarket_mvp/reporting.py**

In [ ]:
%%modulo src/polymarket_mvp/reporting.py
"""Coverage and measured cost reports; estimates remain explicitly labelled."""

import json
import time
from pathlib import Path
from .download import inventory, atomic_json


def progress(root):
    root = Path(root)
    result = []
    for path in sorted((root / 'benchmarks').glob('*/status.json')):
        r = json.loads(path.read_text())
        r['run'] = path.parent.name
        import psutil

        pid = r.get('pid')
        r['pid_alive'] = bool(pid and psutil.pid_exists(pid))
        if r['pid_alive']:
            try:
                r['pid_alive'] = psutil.Process(pid).status() != psutil.STATUS_ZOMBIE
            except psutil.NoSuchProcess:
                r['pid_alive'] = False
        if not r['pid_alive'] and r.get('state') in ['running', 'consolidation']:
            r['observed_state'] = 'stale_process_not_running'
        p = path.parent / 'progress.json'
        if p.exists():
            r['progress'] = json.loads(p.read_text())
        result.append(r)
    return result


REPRESENTATIVE_DAYS = ("2026-02-26", "2026-04-26", "2026-06-06", "2026-06-09")
DISK_RESERVE = 32 * 1024**3


def audit_benchmark(root):
    """Read the exact cohort used for the measured concurrent wall time."""
    root = Path(root)
    cohort_path = root / "parallel4_benchmark.json"
    if not cohort_path.exists():
        raise ValueError(
            "Missing completed concurrent benchmark: parallel4_benchmark.json"
        )
    cohort = json.loads(cohort_path.read_text())
    if not cohort.get("complete"):
        raise ValueError("Concurrent benchmark is incomplete")
    files = []
    for day in REPRESENTATIVE_DAYS:
        folder = root / "benchmarks" / f"{day}_local_parallel4"
        path = folder / "benchmark.json"
        if not path.exists():
            raise ValueError(f"Missing complete-file report for {day}: {path}")
        r = json.loads(path.read_text())
        i, c = r["ingest"], r["consolidation"]
        if (
            not r["complete"]
            or not i["complete"]
            or i["rows_confirmed"] != i["declared_rows"]
        ):
            raise ValueError(f"Unreconciled benchmark rows for {day}")
        timing, stages = i["timings"], c["stage_timings"]
        local_reads = folder / "original_reads.jsonl"
        visits = {}
        if local_reads.exists():
            for line in local_reads.read_text().splitlines():
                event = json.loads(line)
                if event.get("decoder_passes") == 1:
                    group = event["group"]
                    visits[group] = visits.get(group, 0) + 1
        writes = (
            i["resources_after"]["disk_write_bytes"]
            - i["resources_before"]["disk_write_bytes"]
        )
        writes += (
            c["resources"]["disk_write_bytes"]
            - c["resources_before"]["disk_write_bytes"]
        )
        files.append(
            dict(
                date=day,
                report=str(path),
                rows=i["rows_confirmed"],
                original_bytes=r["source_bytes"],
                ingest_seconds=i["seconds"],
                read_decode_seconds=timing["parquet_read_decode_seconds"],
                transform_seconds=timing["transform_catalog_group_seconds"],
                catalog_seconds=timing["catalog_seconds"],
                publish_seconds=timing["write_hash_commit_seconds"],
                repartition_seconds=stages["partition_seconds"],
                aggregate_seconds=stages["aggregate_seconds"],
                consolidation_seconds=c["seconds"],
                verification_seconds=None,
                ingest_events_per_second=i["events_per_second"],
                peak_worker_rss=max(
                    i["resources_after"]["peak_rss"], c["resources"]["peak_rss"]
                ),
                partial_bytes=i["derived_bytes"],
                intermediate_bytes=sum(
                    p.stat().st_size
                    for p in (folder / "merge_partitioned").rglob("*.parquet")
                ),
                final_bytes=c["output_bytes"],
                series_rows=c["series_rows"],
                observed_disk_write_bytes=writes,
                write_amplification=writes / r["source_bytes"],
                decoded_row_group_passes=i["logical_source_passes"],
                duplicate_decodes=(
                    sum(n - 1 for n in visits.values()) if visits else None
                ),
                download=r["download"],
                derived_scan_passes=c["derived_scan_passes"],
                integrity_output_scan_passes=c["integrity_output_scan_passes"],
            )
        )
    if (
        sum(f["rows"] for f in files) != cohort["rows"]
        or sum(f["original_bytes"] for f in files) != cohort["original_bytes"]
    ):
        raise ValueError("Per-file benchmark totals do not match concurrent cohort")
    result = dict(
        cohort=cohort,
        files=files,
        notes=[
            "Catalog is included in Polars transformation; aggregate includes output writing.",
            "Verification is not isolated in these historical reports.",
            "Process disk writes include final output and metadata; not the live temporary peak.",
            "Overlap is accumulated phase time divided by cohort wall time, not a serial speedup.",
        ],
    )
    atomic_json(root / "benchmark_audit.json", result)
    return result


def projection(root, budget_hours=120, preparation=None):
    import shutil
    from . import REVISION

    root = Path(root)
    inv = inventory(root)
    preparation = (
        Path(preparation) if preparation else root / "preflight_full" / "result.json"
    )
    free = shutil.disk_usage(root).free
    try:
        audit = audit_benchmark(root)
        if not preparation.exists():
            raise ValueError(f"Missing end-to-end preparation receipt: {preparation}")
        pilot = json.loads(preparation.read_text())
        if (
            not pilot["complete"]
            or not pilot["training"]["complete"]
            or pilot["source"] not in inv["files"]
        ):
            raise ValueError(f"Incompatible preparation receipt: {preparation}")
        if not pilot["validation"]["windows"] or not pilot["test"]["windows"]:
            raise ValueError("Preparation must evaluate both validation and test")
    except (ValueError, KeyError, TypeError) as error:
        result = dict(
            performance_gate=False,
            storage_gate=False,
            approved_hours=budget_hours,
            error=str(error),
        )
        atomic_json(root / "projection.json", result)
        return result
    cohort, files = audit["cohort"], audit["files"]
    total_bytes = sum(e["bytes"] for e in inv["files"])
    processing = cohort["wall_seconds"] * total_bytes / cohort["original_bytes"]
    final_bytes = 0
    series_rows = 0
    for e in inv["files"]:
        day = e["path"].split("date=")[1][:10]
        if day < "2026-04-13":
            reps = files[:1]
        elif e["bytes"] < 1024**3:
            reps = files[2:3]
        else:
            reps = [files[1], files[3]]
        final_bytes += e["bytes"] * max(
            f["final_bytes"] / f["original_bytes"] for f in reps
        )
        series_rows += e["bytes"] * max(
            f["series_rows"] / f["original_bytes"] for f in reps
        )
    transfer = [
        f["download"]
        for f in files
        if f["download"].get("workers") == 4
        and not f["download"].get("recovered_prefix_bytes", 0)
    ]
    if not transfer:
        result = dict(
            performance_gate=False,
            storage_gate=False,
            error="Missing fresh parallel download measurement",
        )
        atomic_json(root / "projection.json", result)
        return result
    network = total_bytes / min(t["original_bytes"] / t["seconds"] for t in transfer)
    # Model storage scales with observed series density, with a separate density margin.
    scale = series_rows / pilot["consolidated"]["series_rows"]
    census_seconds = pilot["census"]["seconds"] * scale
    eligible = int(pilot["census"]["eligible"] * scale)
    observed = int(pilot["census"]["total_windows"] * scale)
    estimated_splits = dict(
        train=int(eligible * 0.7),
        validation=int(eligible * 0.15),
        test=int(eligible * 0.15),
    )
    tensor_bytes = {k: v * 2562 for k, v in estimated_splits.items()}
    prepared_census = Path(pilot["census"]["index"]).parent
    census_measured_bytes = sum(
        p.stat().st_size for p in prepared_census.glob("*.parquet")
    )
    census_bytes = int(1.15 * census_measured_bytes * scale)
    pack_seconds = sum(pilot["packing_seconds"].values()) * scale
    train_rate = 0.8 * pilot["training"]["windows_per_second"]
    eval_rate = 0.8 * min(
        pilot[s]["windows"] / pilot[s]["seconds"] for s in ("validation", "test")
    )
    training = estimated_splits["train"] / train_rate
    evaluation = (estimated_splits["validation"] + estimated_splits["test"]) / eval_rate
    largest = max(e["bytes"] for e in inv["files"])
    # Admission limits the sum of active per-file reservations, allowing fewer
    # concurrent files when free space falls as verified outputs accumulate.
    scratch_factor = 8.5
    max_scratch = int(largest * scratch_factor)
    manifest_bytes = int(total_bytes * 0.01)
    evaluation_spill_and_results = (
        int(1.2 * max(tensor_bytes["validation"], tensor_bytes["test"])) + 2 * 1024**3
    )
    disk_for_training = (
        free
        - 10 * 1024**3
        - final_bytes
        - manifest_bytes
        - DISK_RESERVE
        - census_bytes
        - evaluation_spill_and_results
    )
    estimated_training_cap = max(
        0, min(estimated_splits["train"], int(disk_for_training / (2818 * 1.1)))
    )
    model_bytes = (
        int(estimated_training_cap * 2818 * 1.1) + evaluation_spill_and_results
    )
    training = estimated_training_cap / train_rate
    projected_storage = int(
        final_bytes
        + manifest_bytes
        + DISK_RESERVE
        + max(max_scratch, census_bytes + model_bytes)
    )
    reserve_time = 0.20 * (processing + census_seconds + pack_seconds)
    total = (
        processing
        + network
        + census_seconds
        + pack_seconds
        + training
        + evaluation
        + reserve_time
    )
    result = dict(
        revision=REVISION,
        estimated_not_executed=True,
        approved_hours=budget_hours,
        projected_hours=total / 3600,
        processing_target_hours=72,
        performance_gate=total <= budget_hours * 3600,
        storage_gate=projected_storage <= free and estimated_training_cap > 0,
        full_file_benchmarks=len(files),
        processing_seconds=processing,
        network_seconds=network,
        census_seconds=census_seconds,
        packing_seconds=pack_seconds,
        training_seconds=training,
        evaluation_seconds=evaluation,
        export_seconds=reserve_time,
        training_windows_per_second=train_rate,
        evaluation_windows_per_second=eval_rate,
        microbatch=pilot["training"]["identity"]["microbatch"],
        workers=cohort["workers"],
        consolidators=cohort["consolidators"],
        estimated_eligible_windows=eligible,
        estimated_split_windows=estimated_splits,
        estimated_tensor_bytes_by_split=tensor_bytes,
        estimated_training_disk_cap=estimated_training_cap,
        estimated_observed_windows=observed,
        measured_preflight_census_bytes=census_measured_bytes,
        estimated_final_bytes=int(final_bytes),
        estimated_manifest_bytes=manifest_bytes,
        estimated_census_bytes=census_bytes,
        estimated_model_stage_bytes=model_bytes,
        estimated_storage_requirement=projected_storage,
        disk_available_bytes=free,
        disk_reserve_bytes=DISK_RESERVE,
        scratch_factor=scratch_factor,
        maximum_single_partition_reservation_bytes=max_scratch,
        max_staging_original_bytes=2
        * sum(
            sorted((e["bytes"] for e in inv["files"]), reverse=True)[
                : cohort["workers"]
            ]
        ),
        preflight_receipt=str(preparation),
        benchmark_audit=str(root / "benchmark_audit.json"),
        assumptions=[
            "Runtime scales by measured concurrent cohort bytes; mixed sizes and disk pressure may change throughput.",
            "Network conservatively added without overlap.",
            "Census density and split sizes are estimates from one exploratory day.",
            "Train selection uses a disk cap; actual minimum is checked against it after the complete census.",
            "Validation and test remain exhaustive; at most one evaluation split is packed at a time.",
        ],
    )
    atomic_json(root / "projection.json", result)
    return result

**src/polymarket_mvp/runner.py**

In [ ]:
%%modulo src/polymarket_mvp/runner.py
"""Persistent benchmark runner; FULL cannot bypass measured feasibility gates."""

import argparse
import json
import os
import time
from pathlib import Path
from .download import inventory, download, atomic_json
from .engine import process, consolidate


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--root', default='runs/mvp')
    p.add_argument('--date', required=True)
    p.add_argument('--mode', choices=['local', 'remote'], default='local')
    p.add_argument('--batch-size', type=int, default=100288)
    p.add_argument('--groups-per-unit', type=int, default=64)
    p.add_argument('--tag', default='v1')
    p.add_argument('--stream-groups', action='store_true')
    p.add_argument('--dictionary-read', action='store_true')
    p.add_argument('--sparse-merge', action='store_true')
    a = p.parse_args()
    root = Path(a.root)
    out = root / 'benchmarks' / f'{a.date}_{a.mode}_{a.tag}'
    out.mkdir(parents=True, exist_ok=True)
    import fcntl

    lock = (out / 'run.lock').open('a')
    fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
    if (out / 'benchmark.json').exists() and json.loads(
        (out / 'benchmark.json').read_text()
    ).get('complete'):
        from .engine import verify_series

        cached = json.loads((out / 'benchmark.json').read_text())
        verify_series(cached['consolidation'])
        print('Verified complete:', out, flush=True)
        return
    t = time.perf_counter()
    atomic_json(
        out / 'status.json',
        dict(
            state='running', pid=os.getpid(), started_utc=time.time(), arguments=vars(a)
        ),
    )
    try:
        inv = inventory(root)
        entry = next(x for x in inv['files'] if a.date in x['path'])
        transfer = None
        source = None
        if a.mode == 'local':
            source, transfer = download(entry, root / 'staging')
        report = process(
            entry,
            source,
            out,
            mode=a.mode,
            batch_size=a.batch_size,
            groups_per_unit=a.groups_per_unit,
            stream_groups=a.stream_groups,
            dictionary_read=a.dictionary_read,
        )
        merged = consolidate(out, sparse_merge=a.sparse_merge)
        atomic_json(
            out / 'benchmark.json',
            dict(
                ingest=report,
                download=transfer,
                consolidation=merged,
                wall_seconds=time.perf_counter() - t,
                source_bytes=entry['bytes'],
                complete=report['complete'],
            ),
        )
        atomic_json(
            out / 'status.json',
            dict(state='complete', pid=os.getpid(), finished_utc=time.time()),
        )
    except BaseException as e:
        atomic_json(
            out / 'status.json',
            dict(state='failed', pid=os.getpid(), error=repr(e), utc=time.time()),
        )
        raise


if __name__ == '__main__':
    main()

**src/polymarket_mvp/storage.py**

In [ ]:
%%modulo src/polymarket_mvp/storage.py
"""Crash-safe publication: immutable Parquet objects precede SQLite confirmation."""

import hashlib
import json
import os
import sqlite3
import uuid
from pathlib import Path

import pyarrow.parquet as pq


def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


class Manifest:
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.db = sqlite3.connect(self.root / 'manifest.sqlite')
        self.db.execute('PRAGMA journal_mode=WAL')
        self.db.execute('PRAGMA synchronous=FULL')
        self.db.executescript('''
        CREATE TABLE IF NOT EXISTS commits(unit TEXT PRIMARY KEY, metadata TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS objects(unit TEXT, path TEXT UNIQUE, sha256 TEXT, rows INTEGER);
        CREATE TABLE IF NOT EXISTS compactions(name TEXT PRIMARY KEY, metadata TEXT NOT NULL);
        CREATE TABLE IF NOT EXISTS replacements(path TEXT PRIMARY KEY, sha256 TEXT NOT NULL, rows INTEGER NOT NULL);
        CREATE TABLE IF NOT EXISTS retired(path TEXT PRIMARY KEY);
        CREATE TABLE IF NOT EXISTS io(id INTEGER PRIMARY KEY, source TEXT, row_group INTEGER,
          operation TEXT, bytes INTEGER, seconds REAL, attempt TEXT);
        ''')

    def confirmed(self, unit):
        return (
            self.db.execute('SELECT 1 FROM commits WHERE unit=?', (unit,)).fetchone()
            is not None
        )

    def publish(self, unit, tables, metadata, fail_at=None):
        if self.confirmed(unit):
            return False
        objects = []
        for kind, table in tables.items():
            name = f'{uuid.uuid4().hex}_{kind}.parquet'
            path = self.root / name
            tmp = path.with_suffix('.tmp')
            from .columnar import write

            write(table, tmp)
            with open(tmp, 'rb') as f:
                os.fsync(f.fileno())
            sha = digest(tmp)
            if pq.ParquetFile(tmp).metadata.num_rows != table.num_rows:
                raise ValueError('Parquet row count mismatch')
            os.replace(tmp, path)  # Publicar antes de confirmar en el manifiesto.
            objects.append((unit, name, sha, table.num_rows))
        fd = os.open(self.root, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)
        if fail_at == 'before_commit':
            raise RuntimeError('Injected crash before manifest commit')
        with self.db:
            self.db.executemany('INSERT INTO objects VALUES (?,?,?,?)', objects)
            self.db.execute(
                'INSERT INTO commits VALUES (?,?)',
                (unit, json.dumps(metadata, sort_keys=True)),
            )
        return True

    def verify(self):
        for path, sha, rows in self.db.execute(
            'SELECT path,sha256,rows FROM objects WHERE path NOT IN (SELECT path FROM retired) UNION ALL SELECT path,sha256,rows FROM replacements'
        ):
            p = self.root / path
            if (
                not p.exists()
                or digest(p) != sha
                or pq.ParquetFile(p).metadata.num_rows != rows
            ):
                raise ValueError(f'Invalid committed object: {path}')
        return self.db.execute('SELECT count(*) FROM commits').fetchone()[0]

    def orphans(self):
        known = {r[0] for r in self.db.execute('SELECT path FROM objects')}
        return sorted(
            p.name for p in self.root.glob('*.parquet') if p.name not in known
        )

    def compact(self, paths, metadata, delete_originals=False):
        """Commit verified replacements before retiring/deleting any source object."""
        objects = []
        for path in paths:
            p = Path(path).resolve()
            sha = digest(p)
            if sha != metadata['sha256'].get(p.name):
                raise ValueError(
                    f'Cannot retire partials: canonical checksum mismatch for {p}'
                )
            objects.append((str(p), sha, pq.ParquetFile(p).metadata.num_rows))
        if sum(rows for _, _, rows in objects) != metadata['series_rows']:
            raise ValueError('Cannot retire partials: canonical row count mismatch')
        self.verify()
        with self.db:
            self.db.executemany(
                'INSERT OR REPLACE INTO replacements VALUES (?,?,?)', objects
            )
            self.db.execute(
                'INSERT OR REPLACE INTO compactions VALUES (?,?)',
                ('final_series', json.dumps(metadata, sort_keys=True)),
            )
            self.db.execute(
                "INSERT OR IGNORE INTO retired SELECT path FROM objects WHERE path LIKE '%_features.parquet'"
            )
        self.verify()
        if delete_originals:
            for (name,) in self.db.execute('SELECT path FROM retired'):
                path = self.root / name
                if not path.resolve().is_relative_to(self.root.resolve()):
                    raise ValueError('Refusing deletion outside owned parts directory')
                path.unlink(missing_ok=True)
        return len(objects)

**src/polymarket_mvp/windows.py**

In [ ]:
%%modulo src/polymarket_mvp/windows.py
"""Complete sparse window census, strictly separated from observable extraction."""

import hashlib
import json
import os
import time
from pathlib import Path
import duckdb
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from .download import atomic_json

SPLITS = {
    'train': (1771718400, 1782000000),
    'validation': (1782000000, 1784160000),
    'test': (1784160000, 1786406400),
}


def build_census(series, root, splits=None, buckets=128):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    splits = splits or SPLITS
    fingerprint = hashlib.sha256(
        json.dumps(
            dict(
                series=[str(Path(p).resolve()) for p in series],
                splits=splits,
                buckets=buckets,
            ),
            sort_keys=True,
        ).encode()
    ).hexdigest()
    marker = root / 'census.json'
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved['identity'] != fingerprint:
            raise ValueError('Census configuration mismatch')
        return saved
    t = time.perf_counter()
    db = duckdb.connect()
    db.execute("SET memory_limit='4GB'")
    db.execute('SET threads=4')
    db.execute('SET temp_directory=?', [str(root / 'spill')])
    db.from_parquet([str(p) for p in series]).create_view('series')
    normalized = root / 'normalized'
    if not (root / 'normalized.json').exists():
        # Global merging handles late events and crossings between original files before windowing.
        select = '''SELECT asset_id,condition_id,second, (second//512)*512 AS block_start,
          (arg_max(struct_pack(v:=spread_last),(event_ts_ms_last,file_ordinal_last,row_position_last))
            FILTER (WHERE event_ts_ms_last IS NOT NULL)).v AS spread,
          sum(events) AS activity,sum(n_v1) AS v1,sum(n_v2) AS v2
          FROM series GROUP BY asset_id,condition_id,second'''
        db.execute(
            f"COPY (SELECT *,count(DISTINCT condition_id) OVER (PARTITION BY asset_id,block_start) AS n_conditions,hash(asset_id,block_start)%{int(buckets)} AS bucket FROM ({select})) TO '{normalized}' (FORMAT PARQUET,PARTITION_BY(bucket),COMPRESSION ZSTD)"
        ).fetchall()
        atomic_json(
            root / 'normalized.json',
            dict(
                identity=fingerprint,
                original_derived_scan_passes=1,
                seconds=time.perf_counter() - t,
            ),
        )
    total = 0
    eligible = 0
    excluded = {}
    split_counts = {k: 0 for k in splits}
    index_writer = None
    schema = pa.schema(
        [
            ('window_id', pa.string()),
            ('asset_id', pa.string()),
            ('condition_id', pa.string()),
            ('start', pa.int64()),
            ('split', pa.string()),
            ('source', pa.string()),
            ('activity', pa.int64()),
            ('observed', pa.int32()),
            ('target', pa.int32()),
            ('reason', pa.string()),
            ('file', pa.string()),
            ('row_group', pa.int32()),
            ('row', pa.int32()),
        ]
    )
    index_tmp = root / 'index.partial.parquet'
    index_writer = pq.ParquetWriter(index_tmp, schema, compression='zstd')
    try:
        for folder in sorted(normalized.glob('bucket=*')):
            outfile = root / (folder.name + '.parquet')
            db.from_parquet(str(folder / '*.parquet')).create_view(
                'bucket_input', replace=True
            )
            q = '''SELECT asset_id,condition_id,block_start,list(second-block_start ORDER BY second) AS offsets,
                list(spread ORDER BY second) AS values,sum(activity)::BIGINT AS activity,
                sum(v1)::BIGINT AS v1,sum(v2)::BIGINT AS v2,max(n_conditions) AS n_conditions
                FROM bucket_input GROUP BY asset_id,condition_id,block_start ORDER BY asset_id,condition_id,block_start'''
            if not outfile.exists():
                tmp = outfile.with_suffix('.tmp')
                db.execute(
                    f"COPY ({q}) TO '{tmp}' (FORMAT PARQUET,COMPRESSION ZSTD,ROW_GROUP_SIZE 16384)"
                ).fetchall()
                os.replace(tmp, outfile)
            p = pq.ParquetFile(outfile)
            for rg in range(p.num_row_groups):
                records = []
                for row, r in enumerate(p.read_row_group(rg).to_pylist()):
                    total += 1
                    start = r['block_start']
                    split = next(
                        (
                            s
                            for s, (a, b) in splits.items()
                            if a <= start and start + 512 <= b
                        ),
                        None,
                    )
                    offsets = np.asarray(r['offsets'], dtype=np.int32)
                    values = np.asarray(r['values'], dtype=np.float64)
                    good = np.isfinite(values)
                    obs = np.zeros(512, dtype=bool)
                    obs[offsets[good]] = True
                    patches = np.flatnonzero(obs.reshape(64, 8).all(1))
                    target = int(patches[-1]) * 8 if len(patches) >= 2 else -1
                    reason = 'eligible'
                    if split is None:
                        reason = 'split_boundary_or_outside'
                    elif r['condition_id'] is None:
                        reason = 'missing_condition'
                    elif r['n_conditions'] > 1:
                        reason = 'conflicting_condition'
                    elif target < 0:
                        reason = 'insufficient_complete_patches'
                    if reason == 'eligible':
                        eligible += 1
                        split_counts[split] += 1
                    else:
                        excluded[reason] = excluded.get(reason, 0) + 1
                    source = (
                        'mixed'
                        if r['v1'] and r['v2']
                        else 'v1' if r['v1'] else 'v2' if r['v2'] else 'unknown'
                    )
                    wid = hashlib.sha256(
                        f'{r["asset_id"]}|{r["condition_id"]}|{start}'.encode()
                    ).hexdigest()
                    records.append(
                        dict(
                            window_id=wid,
                            asset_id=r['asset_id'],
                            condition_id=r['condition_id'],
                            start=start,
                            split=split,
                            source=source,
                            activity=r['activity'],
                            observed=int(obs.sum()),
                            target=target,
                            reason=reason,
                            file=str(outfile.resolve()),
                            row_group=rg,
                            row=row,
                        )
                    )
                index_writer.write_table(pa.Table.from_pylist(records, schema=schema))
    finally:
        index_writer.close()
        db.close()
    os.replace(index_tmp, root / 'index.parquet')
    report = dict(
        identity=fingerprint,
        total_windows=total,
        eligible=eligible,
        excluded=excluded,
        split_counts=split_counts,
        seconds=time.perf_counter() - t,
        source_derived_passes=1,
        normalized_passes=1,
        window_records_passes=1,
        splits=splits,
        index=str((root / 'index.parquet').resolve()),
        final_independent_test=splits == SPLITS,
    )
    atomic_json(marker, report)
    return report


# Selección de entrenamiento


def select_training(census_root, budget, seed=2026):
    root = Path(census_root)
    from .storage import digest

    if (root / 'selection.json').exists():
        saved = json.loads((root / 'selection.json').read_text())
        if saved.get('seed') != seed:
            raise ValueError('Frozen selection seed mismatch')
        if saved.get('selection_sha256') != digest(root / 'selection.parquet'):
            raise ValueError(
                'Frozen selection integrity mismatch; use a new experiment for legacy selections'
            )
        return saved
    db = duckdb.connect()
    db.execute("SET memory_limit='3GB'")
    db.from_parquet(str(root / 'index.parquet')).create_view('census')
    edges = db.execute(
        "SELECT quantile_cont(activity,[0.25,0.5,0.75]) FROM census WHERE split='train' AND reason='eligible'"
    ).fetchall()[0][0]
    if edges is None:
        raise ValueError('No eligible training windows')
    q = f'''SELECT *,CAST(start//86400 AS BIGINT) AS day,
       CASE WHEN activity<={edges[0]} THEN 0 WHEN activity<={edges[1]} THEN 1 WHEN activity<={edges[2]} THEN 2 ELSE 3 END AS activity_bin,
       sha256(window_id||'{int(seed)}') AS draw FROM census WHERE split='train' AND reason='eligible' '''
    db.execute('CREATE TEMP TABLE candidates AS ' + q).fetchall()
    db.execute('''CREATE TEMP TABLE ranked AS SELECT *,
       row_number() OVER(PARTITION BY day,condition_id,source,activity_bin ORDER BY draw,window_id) AS stratum_rank,
       row_number() OVER(PARTITION BY asset_id ORDER BY draw,window_id) AS asset_rank
       FROM candidates''').fetchall()
    minimum, total = db.execute(
        'SELECT count(*) FILTER(WHERE stratum_rank=1 OR asset_rank=1),count(*) FROM ranked'
    ).fetchall()[0]
    report = dict(
        minimum_windows=minimum,
        eligible_train=total,
        budget_windows=int(budget),
        seed=seed,
        activity_edges=edges,
        cardinality_gate=minimum <= budget,
    )
    atomic_json(root / 'selection_budget.json', report)
    if minimum > budget:
        alternative = db.execute('''SELECT count(*) FROM (
          SELECT *,row_number() OVER(PARTITION BY day ORDER BY draw,window_id) AS d,
            row_number() OVER(PARTITION BY condition_id ORDER BY draw,window_id) AS c,
            row_number() OVER(PARTITION BY source,activity_bin ORDER BY draw,window_id) AS s
          FROM ranked) WHERE d=1 OR c=1 OR s=1 OR asset_rank=1''').fetchall()[0][0]
        report.update(
            conflict_windows=minimum - budget,
            alternative_policy='deterministic marginal coverage of every date, condition, asset and source/activity bin; no condition-day cross-product guarantee',
            alternative_minimum_windows=alternative,
            alternative_fits=alternative <= budget,
            requires_user_decision=True,
        )
        atomic_json(root / 'selection_budget.json', report)
        db.close()
        raise RuntimeError(f'Minimum coverage exceeds measured budget: {report}')
    # Mandatory coverage first; remaining quota uses sqrt(stratum size), then deterministic hash draws.
    db.execute(
        '''CREATE TEMP TABLE mandatory AS SELECT * FROM ranked WHERE stratum_rank=1 OR asset_rank=1'''
    ).fetchall()
    db.execute(
        '''CREATE TEMP TABLE remaining AS SELECT * FROM ranked WHERE stratum_rank<>1 AND asset_rank<>1'''
    ).fetchall()
    extra = min(total, budget) - minimum
    db.execute(
        '''CREATE TEMP TABLE allocation AS SELECT day,condition_id,source,activity_bin,count(*) AS n,sqrt(count(*)) AS weight
                  FROM remaining GROUP BY ALL'''
    ).fetchall()
    # Weighted deterministic priority gives balanced representation without replacement.
    db.execute('''CREATE TEMP TABLE chosen AS SELECT * FROM mandatory''').fetchall()
    if extra:
        db.execute(
            f'''INSERT INTO chosen SELECT r.* FROM remaining r JOIN allocation a USING(day,condition_id,source,activity_bin)
          ORDER BY r.stratum_rank/a.weight,r.draw,r.window_id LIMIT {int(extra)}'''
        ).fetchall()
    dest = root / 'selection.parquet'
    db.execute(
        f"COPY (SELECT * FROM chosen ORDER BY draw,window_id) TO '{dest}' (FORMAT PARQUET,COMPRESSION ZSTD)"
    ).fetchall()
    db.execute(
        f"COPY (SELECT day,condition_id,source,activity_bin,count(*) AS selected,count(DISTINCT asset_id) AS assets FROM chosen GROUP BY ALL) TO '{root/'selection_distribution.parquet'}' (FORMAT PARQUET,COMPRESSION ZSTD)"
    ).fetchall()
    distribution = db.execute(
        'SELECT source,activity_bin,count(*),count(DISTINCT asset_id),count(DISTINCT condition_id) FROM chosen GROUP BY ALL'
    ).fetchall()
    report.update(
        selection_sha256=digest(root / 'selection.parquet'),
        selected=min(total, budget),
        excluded_budget=total - min(total, budget),
        distribution=distribution,
        epoch_definition='one pass over frozen selection',
    )
    atomic_json(root / 'selection.json', report)
    db.close()
    return report


# Preparación de ventanas


def pack_windows(
    census_root, output, split, selected=False, deadline=None, reserve_bytes=0
):
    """Stream compact locators and write memmaps; never materialize the index in RAM."""
    from .storage import digest
    import fcntl, shutil, time

    root = Path(census_root)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    with (output / 'pack.lock').open('a') as lock:
        fcntl.flock(lock.fileno(), fcntl.LOCK_EX)
        index_path = root / ('selection.parquet' if selected else 'index.parquet')
        index_sha = digest(index_path)
        receipt = output / 'pack.json'
        if receipt.exists():
            old = json.loads(receipt.read_text())
            if old.get('input_index_sha256') != index_sha or old.get('split') != split:
                raise ValueError('Pack input changed; use a new output directory')
            for name, sha in old['hashes'].items():
                if digest(output / name) != sha:
                    raise ValueError(f'Corrupt pack {name}')
            return old['windows']
        db = duckdb.connect()
        db.execute("SET memory_limit='8GB'")
        db.execute("SET threads=2")
        db.execute("SET preserve_insertion_order=false")
        db.execute('SET temp_directory=?', [str(output / 'spill')])
        db.from_parquet(str(index_path)).create_view('idx')
        n = db.execute(
            "SELECT count(*) FROM idx WHERE reason='eligible' AND split=?", [split]
        ).fetchall()[0][0]
        if not n:
            db.close()
            raise ValueError('No windows to pack')
        required = n * (2562 + 256)
        free = shutil.disk_usage(output).free
        if free < required + reserve_bytes:
            db.close()
            raise OSError(
                f"Insufficient disk for {split}: {n} windows require {required} bytes plus {reserve_bytes} reserve, {free} free"
            )
        x = np.lib.format.open_memmap(
            output / 'x.npy', mode='w+', dtype=np.float32, shape=(n, 512)
        )
        observed = np.lib.format.open_memmap(
            output / 'observed.npy', mode='w+', dtype=np.uint8, shape=(n, 512)
        )
        target = np.lib.format.open_memmap(
            output / 'target.npy', mode='w+', dtype=np.int16, shape=(n,)
        )
        query = "SELECT window_id,asset_id,condition_id,start,target,source,activity,observed,split,file,row_group,row FROM idx WHERE reason='eligible' AND split=? ORDER BY file,row_group,row"
        reader = db.sql(query, params=[split]).to_arrow_reader(batch_size=16384)
        writer = None
        cached_key = None
        cached = None
        i = 0
        rg_reads = 0
        try:
            for batch in reader:
                if deadline is not None and time.time() >= deadline:
                    raise TimeoutError(f"Budget reached while packing {split}")
                table = pa.Table.from_batches([batch])
                if writer is None:
                    writer = pq.ParquetWriter(
                        output / 'metadata.partial.parquet',
                        table.schema,
                        compression='zstd',
                    )
                writer.write_table(table)
                for r in table.to_pylist():
                    key = (r['file'], r['row_group'])
                    if key != cached_key:
                        cached = (
                            pq.ParquetFile(key[0])
                            .read_row_group(key[1], columns=['offsets', 'values'])
                            .to_pylist()
                        )
                        cached_key = key
                        rg_reads += 1
                    record = cached[r['row']]
                    offset = np.asarray(record['offsets'])
                    v = np.asarray(record['values'], dtype=np.float32)
                    good = np.isfinite(v)
                    x[i] = 0
                    observed[i] = 0
                    x[i, offset[good]] = v[good]
                    observed[i, offset[good]] = 1
                    target[i] = r['target']
                    i += 1
        finally:
            reader.close()
            if writer:
                writer.close()
            db.close()
        if i != n:
            raise ValueError('Pack cardinality mismatch')
        x.flush()
        observed.flush()
        target.flush()
        os.replace(output / 'metadata.partial.parquet', output / 'metadata.parquet')
        hashes = {
            name: digest(output / name)
            for name in ['x.npy', 'observed.npy', 'target.npy', 'metadata.parquet']
        }
        for name in hashes:
            with (output / name).open('rb') as f:
                os.fsync(f.fileno())
        atomic_json(
            receipt,
            dict(
                windows=n,
                split=split,
                selected=selected,
                input_index_sha256=index_sha,
                index_passes=2,
                window_row_groups_read=rg_reads,
                window_row_groups_read_once=True,
                hashes=hashes,
                pack_integrity_scan_passes=1,
                input_integrity_scan_passes=1,
                max_locator_batch=16384,
            ),
        )
        return n


def repartition_census(source_root, destination, splits):
    """Change experimental temporal splits using the index, without rereading features."""
    source_root = Path(source_root)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    db = duckdb.connect()
    db.from_parquet(str(source_root / 'index.parquet')).create_view('old')
    cases = ' '.join(
        f"WHEN start>={int(a)} AND start+512<={int(b)} THEN '{s}'"
        for s, (a, b) in splits.items()
    )
    q = f'''SELECT * EXCLUDE(split,reason),CASE {cases} END AS split,
      CASE WHEN NOT ({' OR '.join(f'(start>={a} AND start+512<={b})' for a,b in splits.values())}) THEN 'split_boundary_or_outside'
        WHEN condition_id IS NULL THEN 'missing_condition'
        WHEN count(DISTINCT condition_id) OVER(PARTITION BY asset_id,start)>1 THEN 'conflicting_condition'
        WHEN target<0 THEN 'insufficient_complete_patches' ELSE 'eligible' END AS reason FROM old'''
    path = destination / 'index.parquet'
    db.execute(f"COPY ({q}) TO '{path}' (FORMAT PARQUET,COMPRESSION ZSTD)").fetchall()
    db.from_parquet(str(path)).create_view('new')
    excluded = dict(
        db.execute(
            "SELECT reason,count(*) FROM new WHERE reason<>'eligible' GROUP BY reason"
        ).fetchall()
    )
    counts = dict(
        db.execute(
            "SELECT split,count(*) FROM new WHERE reason='eligible' GROUP BY split"
        ).fetchall()
    )
    r = dict(
        total_windows=db.execute('SELECT count(*) FROM new').fetchall()[0][0],
        eligible=sum(counts.values()),
        excluded=excluded,
        split_counts=counts,
        splits=splits,
        source_derived_passes=0,
        index_relabel_passes=1,
        final_independent_test=False,
        source_census=str(source_root.resolve()),
        index=str(path.resolve()),
    )
    atomic_json(destination / 'census.json', r)
    db.close()
    return r


# Keep the previous implementation available for measured comparisons; new runs use
# the streaming calendar frontier and vectorized SQL census below.
build_census_legacy = build_census


def build_census(
    series, root, splits=None, buckets=16, deadline=None, canonical_unique=False
):
    import shutil
    import pyarrow.compute as pc

    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    splits = splits or SPLITS
    import fcntl

    lock = (root / 'census.lock').open('a')
    fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
    marker = root / 'census.json'
    identity = hashlib.sha256(
        json.dumps(
            dict(
                series=[str(Path(p).resolve()) for p in series],
                splits=splits,
                buckets=buckets,
                canonical_unique=canonical_unique,
                version=4,
            ),
            sort_keys=True,
        ).encode()
    ).hexdigest()
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved.get('identity') != identity:
            raise ValueError('Census configuration mismatch')
        return saved
    started = time.perf_counter()
    db = duckdb.connect()
    db.execute("SET memory_limit='3GB'")
    db.execute('SET threads=4')
    db.execute('SET temp_directory=?', [str(root / 'spill')])
    stage = root / 'normalized_v2'
    if stage.exists():
        shutil.rmtree(stage)  # only unconfirmed, regenerable census staging
    stage.mkdir(exist_ok=True)
    schema = pa.schema(
        [
            ('window_id', pa.string()),
            ('asset_id', pa.string()),
            ('condition_id', pa.string()),
            ('start', pa.int64()),
            ('split', pa.string()),
            ('source', pa.string()),
            ('activity', pa.int64()),
            ('observed', pa.int32()),
            ('target', pa.int32()),
            ('reason', pa.string()),
            ('file', pa.string()),
            ('row_group', pa.int32()),
            ('row', pa.int32()),
        ]
    )
    writer = pq.ParquetWriter(
        root / 'index.partial.parquet', schema, compression='zstd'
    )
    # Footer ranges determine a safe completion frontier, even if nominal file dates
    # are wrong. A future file cannot contribute seconds earlier than its own minimum.
    sources = []
    wanted = [
        'asset_id',
        'condition_id',
        'second',
        'spread_last',
        'event_ts_ms_last',
        'file_ordinal_last',
        'row_position_last',
        'events',
        'n_v1',
        'n_v2',
    ]
    for path in series:
        p = pq.ParquetFile(path)
        j = p.schema_arrow.names.index('second')
        mins = []
        for g in range(p.num_row_groups):
            stat = p.metadata.row_group(g).column(j).statistics
            if stat is None or not stat.has_min_max:
                raise ValueError('Generated series require timestamp statistics')
            mins.append(stat.min)
        if mins:
            sources.append((min(mins), str(path)))
    sources.sort()
    total = 0
    eligible = 0
    excluded = {}
    split_counts = {s: 0 for s in splits}
    completed_days = set()
    split_sql = (
        'CASE '
        + ' '.join(
            f"WHEN block_start>={int(a)} AND block_start+512<={int(b)} THEN '{s}'"
            for s, (a, b) in splits.items()
        )
        + ' END'
    )
    reads = root / 'derived_reads.jsonl'
    pending = set()
    chunk = 0

    def finalize(day):
        nonlocal total, eligible
        folders = sorted(stage.glob(f'day={day}/bucket=*'))
        for folder in folders:
            if deadline is not None and time.time() >= deadline:
                raise TimeoutError('Approved budget reached at census bucket boundary')
            db.from_parquet(str(folder / '*.parquet')).create_view('raw', replace=True)
            # Only compact observations for this day/bucket are grouped; no global
            # list aggregation and no rereading the full feature list per bucket.
            merged = '''SELECT asset_id,condition_id,second,(second//512)*512 AS block_start,
                (arg_max(struct_pack(v:=spread_last),(event_ts_ms_last,file_ordinal_last,row_position_last))
                 FILTER(WHERE event_ts_ms_last IS NOT NULL)).v AS spread,
                sum(events)::BIGINT AS activity,sum(n_v1)::BIGINT AS v1,sum(n_v2)::BIGINT AS v2
                FROM raw GROUP BY asset_id,condition_id,second'''
            if canonical_unique:
                merged = '''SELECT asset_id,condition_id,second,(second//512)*512 AS block_start,
                    spread_last AS spread,events::BIGINT AS activity,n_v1::BIGINT AS v1,n_v2::BIGINT AS v2 FROM raw'''
            db.execute('CREATE OR REPLACE TEMP TABLE merged AS ' + merged).fetchall()
            db.execute('''CREATE OR REPLACE TEMP TABLE patches AS
                SELECT asset_id,condition_id,block_start,second//8*8-block_start AS patch,
                    count(*) FILTER(WHERE spread IS NOT NULL AND isfinite(spread)) AS observed
                FROM merged GROUP BY ALL''').fetchall()
            db.execute('''CREATE OR REPLACE TEMP TABLE targets AS
                SELECT asset_id,condition_id,block_start,
                   CASE WHEN count(*) FILTER(WHERE observed=8)>=2 THEN max(patch) FILTER(WHERE observed=8) ELSE -1 END::INTEGER AS target
                FROM patches GROUP BY asset_id,condition_id,block_start''').fetchall()
            # Census metadata covers every observed block. Arrays are materialized
            # only for eligible windows; all other channels remain in feature Parquet.
            windows = '''SELECT asset_id,condition_id,block_start,
                sum(activity)::BIGINT AS activity,sum(v1) AS v1,sum(v2) AS v2,
                count(*) FILTER(WHERE spread IS NOT NULL AND isfinite(spread))::INTEGER AS observed
                FROM merged GROUP BY asset_id,condition_id,block_start'''
            q = f'''WITH w AS ({windows}), j AS (
                SELECT w.*,t.target,count(DISTINCT w.condition_id) OVER(PARTITION BY w.asset_id,w.block_start) AS n_conditions
                FROM w JOIN targets t ON w.asset_id=t.asset_id AND w.condition_id IS NOT DISTINCT FROM t.condition_id AND w.block_start=t.block_start),
                s AS (SELECT *,{split_sql} AS split FROM j)
                SELECT *,sha256(asset_id||'|'||coalesce(condition_id,'None')||'|'||CAST(block_start AS VARCHAR)) AS window_id,
                  CASE WHEN v1>0 AND v2>0 THEN 'mixed' WHEN v1>0 THEN 'v1' WHEN v2>0 THEN 'v2' ELSE 'unknown' END AS source,
                  CASE WHEN split IS NULL THEN 'split_boundary_or_outside' WHEN condition_id IS NULL THEN 'missing_condition'
                       WHEN n_conditions>1 THEN 'conflicting_condition' WHEN target<0 THEN 'insufficient_complete_patches' ELSE 'eligible' END AS reason
                FROM s'''
            db.execute('CREATE OR REPLACE TEMP TABLE window_meta AS ' + q).fetchall()
            data_query = '''SELECT w.asset_id,w.condition_id,w.block_start,w.window_id,
                list(m.second-m.block_start ORDER BY m.second) AS offsets,list(m.spread ORDER BY m.second) AS values
                FROM merged m JOIN window_meta w ON m.asset_id=w.asset_id AND m.condition_id IS NOT DISTINCT FROM w.condition_id
                    AND m.block_start=w.block_start WHERE w.reason='eligible'
                GROUP BY w.asset_id,w.condition_id,w.block_start,w.window_id ORDER BY w.asset_id,w.condition_id,w.block_start'''
            output = root / f'day={day}_{folder.name}.parquet'
            tmp = output.with_suffix('.tmp')
            db.execute(
                f"COPY ({data_query}) TO '{tmp}' (FORMAT PARQUET,COMPRESSION ZSTD,ROW_GROUP_SIZE 16384)"
            ).fetchall()
            os.replace(tmp, output)
            db.execute(
                'CREATE OR REPLACE TEMP TABLE locs(window_id VARCHAR,file VARCHAR,rg INTEGER,pos INTEGER)'
            ).fetchall()
            p = pq.ParquetFile(output)
            for rg in range(p.num_row_groups):
                loc = p.read_row_group(rg, columns=['window_id'])
                n = loc.num_rows
                loc = (
                    loc.append_column('file', pa.array([str(output.resolve())] * n))
                    .append_column('rg', pa.array(np.full(n, rg, dtype=np.int32)))
                    .append_column('pos', pa.array(np.arange(n, dtype=np.int32)))
                )
                db.from_arrow(loc).create_view('location_chunk', replace=True)
                db.execute('INSERT INTO locs SELECT * FROM location_chunk').fetchall()
            index_query = '''SELECT w.window_id,w.asset_id,w.condition_id,w.block_start AS start,w.split,w.source,w.activity,
                w.observed,w.target,w.reason,l.file,coalesce(l.rg,-1)::INTEGER AS row_group,coalesce(l.pos,-1)::INTEGER AS row
                FROM window_meta w LEFT JOIN locs l USING(window_id) ORDER BY w.asset_id,w.condition_id,w.block_start'''
            index_reader = db.sql(index_query).to_arrow_reader(batch_size=16384)
            try:
                for batch in index_reader:
                    table = pa.Table.from_batches([batch]).cast(schema)
                    n = table.num_rows
                    total += n
                    for reason in pc.value_counts(table['reason']).to_pylist():
                        if reason['values'] == 'eligible':
                            eligible += reason['counts']
                        else:
                            excluded[reason['values']] = (
                                excluded.get(reason['values'], 0) + reason['counts']
                            )
                    for split in splits:
                        split_counts[split] += int(
                            pc.sum(
                                pc.cast(
                                    pc.and_(
                                        pc.equal(table['reason'], 'eligible'),
                                        pc.fill_null(
                                            pc.equal(table['split'], split), False
                                        ),
                                    ),
                                    pa.int64(),
                                )
                            ).as_py()
                            or 0
                        )
                    writer.write_table(table)
            finally:
                index_reader.close()
            # This staging directory is generated by this function, never an original.
            shutil.rmtree(folder)
        completed_days.add(day)

    try:
        for number, (minimum, path) in enumerate(sources):
            if deadline is not None and time.time() >= deadline:
                raise TimeoutError('Approved budget reached at census source boundary')
            p = pq.ParquetFile(path)
            for batch in p.iter_batches(batch_size=500000, columns=wanted):
                d = __import__('polars').from_arrow(batch)
                import polars as pl

                d = d.with_columns(
                    ((pl.col('second') // 512 * 512) // 86400).alias('_day'),
                    (pl.col('asset_id').hash(seed=2026) % buckets).alias('_bucket'),
                )
                for (day, bucket), part in d.partition_by(
                    ['_day', '_bucket'], as_dict=True
                ).items():
                    day = int(day)
                    bucket = int(bucket)
                    if day in completed_days:
                        raise ValueError('Invalid timestamp frontier')
                    pending.add(day)
                    folder = stage / f'day={day}' / f'bucket={bucket}'
                    folder.mkdir(parents=True, exist_ok=True)
                    pq.write_table(
                        part.drop('_day', '_bucket').to_arrow(),
                        folder / f'chunk_{chunk:09d}.parquet',
                        compression='zstd',
                        compression_level=1,
                    )
                    chunk += 1
            with reads.open('a') as f:
                f.write(
                    json.dumps(
                        dict(
                            operation='census_source',
                            path=path,
                            pass_number=1,
                            rows=p.metadata.num_rows,
                        )
                    )
                    + '\n'
                )
            frontier = (
                (sources[number + 1][0] // 512 * 512) // 86400
                if number + 1 < len(sources)
                else float('inf')
            )
            for day in sorted(x for x in pending if x < frontier):
                finalize(day)
                pending.remove(day)
        writer.close()
        os.replace(root / 'index.partial.parquet', root / 'index.parquet')
    except BaseException:
        writer.close()
        raise
    finally:
        db.close()
        lock.close()
    r = dict(
        canonical_unique=canonical_unique,
        identity=identity,
        total_windows=total,
        eligible=eligible,
        excluded=excluded,
        split_counts=split_counts,
        splits=splits,
        seconds=time.perf_counter() - started,
        source_derived_passes=1,
        normalized_passes=1,
        index_projection_passes=1,
        source_footer_passes=1,
        window_universe='observed asset-condition blocks only; blocks without events are not enumerated',
        staging_policy='safe timestamp frontier; normalized days reclaimed',
        index=str((root / 'index.parquet').resolve()),
        final_independent_test=splits == SPLITS,
    )
    atomic_json(marker, r)
    return r

**src/polymarket_mvp/worker.py**

In [ ]:
%%modulo src/polymarket_mvp/worker.py
"""Isolated stages release allocator and writer metadata between large files."""

import argparse, json
from pathlib import Path
from .engine import process, consolidate
from .download import atomic_json

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('stage', choices=['ingest', 'consolidate'])
    p.add_argument('--out', required=True)
    p.add_argument('--source')
    p.add_argument('--entry')
    p.add_argument('--stream-groups', action='store_true')
    p.add_argument('--deadline', type=float)
    p.add_argument('--groups-per-unit', type=int, default=16)
    p.add_argument('--dictionary-read', action='store_true')
    p.add_argument('--sparse-merge', action='store_true')
    a = p.parse_args()
    root = Path(a.out)
    if a.stage == 'ingest':
        r = process(
            json.loads(a.entry),
            a.source,
            root,
            batch_size=262144,
            groups_per_unit=a.groups_per_unit,
            stream_groups=a.stream_groups,
            deadline=a.deadline,
            dictionary_read=a.dictionary_read,
        )
    else:
        r = consolidate(root, deadline=a.deadline, sparse_merge=a.sparse_merge)
    atomic_json(root / (a.stage + '_worker_result.json'), r)

**herramientas/inventory_footers.py**

In [ ]:
%%modulo herramientas/inventory_footers.py
"""Concurrent footer-only census, independent of fused event ingestion."""
import json,time,hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pyarrow as pa,pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from polymarket_mvp import REVISION,REPOSITORY
from polymarket_mvp.features import COLUMNS
from polymarket_mvp.download import inventory,atomic_json
root=Path('runs/mvp');dest=root/'metadata_inventory';dest.mkdir(exist_ok=True);started=time.perf_counter()
def one(entry):
    day=entry['path'].split('date=')[1][:10];summary=dest/(day+'.json')
    if summary.exists():return json.loads(summary.read_text())
    fetched=[];t=time.perf_counter()
    with HfFileSystem().open(f'datasets/{REPOSITORY}@{REVISION}/'+entry['path'],'rb',block_size=65536,cache_type='none') as f:
        original=f._fetch_range
        def fetch(a,b):
            t=time.perf_counter();data=original(a,b);fetched.append(dict(start=a,end=b,bytes=len(data),seconds=time.perf_counter()-t));return data
        f._fetch_range=fetch;f.cache.fetcher=fetch;p=pq.ParquetFile(f)
        indices={p.schema.column(i).path:i for i in range(len(p.schema))};rg=[];compressed={c:0 for c in COLUMNS}
        for g in range(p.num_row_groups):
            m=p.metadata.row_group(g);stat=m.column(indices['event_ts_ms']).statistics
            amounts={c:m.column(indices[c]).total_compressed_size for c in COLUMNS}
            for c,n in amounts.items():compressed[c]+=n
            rg.append(dict(row_group=g,rows=m.num_rows,min_ts_ms=stat.min if stat and stat.has_min_max else None,
                max_ts_ms=stat.max if stat and stat.has_min_max else None,selected_compressed_bytes=sum(amounts.values())))
        pq.write_table(pa.Table.from_pylist(rg),dest/(day+'_row_groups.parquet'),compression='zstd')
        r=dict(entry,revision=REVISION,rows=p.metadata.num_rows,row_groups=p.num_row_groups,footer_bytes=p.metadata.serialized_size,
            schema=str(p.schema_arrow),selected_compressed_bytes=compressed,metadata_payload_bytes=sum(x['bytes'] for x in fetched),
            metadata_requests=fetched,seconds=time.perf_counter()-t,original_data_decoder_passes=0)
        if sum(x['rows'] for x in rg)!=r['rows']:raise ValueError('Metadata row sum mismatch')
    atomic_json(summary,r);print(day,r['rows'],r['metadata_payload_bytes'],flush=True);return r
entries=inventory(root)['files']
with ThreadPoolExecutor(max_workers=4) as pool:results=list(pool.map(one,entries))
r=dict(revision=REVISION,files=results,file_count=len(results),original_bytes=sum(x['bytes'] for x in results),
    declared_rows=sum(x['rows'] for x in results),row_groups=sum(x['row_groups'] for x in results),
    metadata_payload_bytes=sum(x['metadata_payload_bytes'] for x in results),seconds=time.perf_counter()-started,
    scope='footers only; does not claim original events processed',data_decoder_passes=0)
atomic_json(root/'inventory_enriched.json',r);print('INVENTORY_COMPLETE',r['declared_rows'],r['row_groups'],flush=True)

**herramientas/certify_daily_partitioning.py**

In [ ]:
%%modulo herramientas/certify_daily_partitioning.py
import datetime,json,hashlib
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow.compute as pc
from polymarket_mvp.download import atomic_json
root=Path('runs/mvp');inventory=json.loads((root/'inventory_enriched.json').read_text());bad=[]
for f in inventory['files']:
    day=f['path'].split('date=')[1][:10];start=(datetime.date.fromisoformat(day)-datetime.date(1970,1,1)).days*86400000
    t=pq.read_table(root/'metadata_inventory'/f'{day}_row_groups.parquet',columns=['min_ts_ms','max_ts_ms'])
    lo=pc.min(t['min_ts_ms']).as_py();hi=pc.max(t['max_ts_ms']).as_py()
    if lo is None or hi is None or lo<start or hi>=start+86400000:bad.append(dict(path=f['path'],minimum=lo,maximum=hi))
r=dict(revision=inventory['revision'],files=164,disjoint_utc_day_timestamp_ranges=not bad,exceptions=bad,
    inventory_sha256=hashlib.sha256((root/'inventory_enriched.json').read_bytes()).hexdigest(),
    proof='Distinct original UTC days plus per-file canonical GROUP BY(asset,condition,second) imply globally unique canonical keys')
atomic_json(root/'canonical_day_certificate.json',r);print(r,flush=True)

**herramientas/preflight_full.py**

In [ ]:
%%modulo herramientas/preflight_full.py
"""Run the release pipeline on a complete original with exploratory UTC splits."""
import json
import sys
import time
from pathlib import Path

from polymarket_mvp.download import atomic_json, download, inventory
from polymarket_mvp.engine import consolidate, process, verify_series
from polymarket_mvp.model import WindowDataset, evaluate, train, verify_adapter
from polymarket_mvp.storage import digest
from polymarket_mvp.windows import build_census, pack_windows, select_training


def main(root):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for the measured preflight")
    output = root / "preflight_full"
    output.mkdir(parents=True, exist_ok=True)
    entry = next(e for e in inventory(root)["files"] if "date=2026-06-06" in e["path"])
    source, network = download(entry, root / "staging")
    if digest(source) != entry["sha256"]:
        raise ValueError("Original SHA256 mismatch")
    ingested = process(entry, source, output / "partition", groups_per_unit=16,
                       batch_size=262144, dictionary_read=True)
    consolidated = consolidate(output / "partition", sparse_merge=True)
    verify_series(consolidated)
    if ingested["rows_confirmed"] != consolidated["events"] + ingested.get("missing_key", 0):
        from polymarket_mvp.storage import Manifest
        manifest = Manifest(output / "partition" / "parts")
        missing = sum(json.loads(r[0])["missing_key"] for r in manifest.db.execute("SELECT metadata FROM commits"))
        manifest.db.close()
        assert ingested["rows_confirmed"] == consolidated["events"] + missing
    splits = {"train": (1780704000, 1780788224), "validation": (1780788224, 1780789248),
              "test": (1780789248, 1780790400)}
    census = build_census(consolidated["outputs"], output / "census", splits=splits, canonical_unique=True)
    selection = select_training(output / "census", census["split_counts"]["train"])
    timings = {}
    for split in splits:
        start = time.perf_counter()
        pack_windows(output / "census", output / split, split, selected=split == "train")
        timings[split] = time.perf_counter() - start
    import numpy as np
    data = WindowDataset(output / "train")
    for i in range(len(data)):
        row = data[i]
        target = int(data.target[i])
        if (row["x"].shape != (1, 512) or row["x"][..., target:].any()
                or row["mask"][target:].any()
                or not np.array_equal(row["y"].numpy(), data.x[i, target:target+8])):
            raise ValueError(f"Invalid target masking in window {i}")
    model_dir = output / "model"
    if not (model_dir / "checkpoint.json").exists():
        _, first = train(output / "train", model_dir, microbatch=256, max_steps=20)
        assert not first["complete"]
    model, training = train(output / "train", model_dir, microbatch=256, resume=True)
    assert training["complete"]
    adapter = verify_adapter(model_dir)
    validation = evaluate(output / "validation", model, output / "evaluation_validation")
    test = evaluate(output / "test", model, output / "evaluation_test", calibration=validation["calibration"])
    atomic_json(output / "result.json", dict(complete=True, exploratory=True, source=entry, download=network,
        ingest=ingested, consolidated=consolidated, census=census, selection=selection, packing_seconds=timings,
        training=training, adapter=adapter, validation=validation, test=test))


if __name__ == "__main__":
    main(Path(sys.argv[1]).resolve())

**tests/mvp/test_engine.py**

In [ ]:
%%modulo tests/mvp/test_engine.py
import tempfile
import unittest
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from polymarket_mvp.engine import process,consolidate

class EngineTests(unittest.TestCase):
    def test_streamed_groups_consolidation_and_retry(self):
        rows=[dict(asset_id='a',condition_id='c',event_ts_ms=1000+i,event_type='price_change',best_bid=.4+i*.001,
                   best_ask=.6,bids_json=None,asks_json=None,source_archive='v2',size=1.,price=.5,side='BUY') for i in range(30)]
        table=pa.Table.from_pylist(rows)
        for name in ['bids_json','asks_json']:
            table=table.set_column(table.schema.get_field_index(name),name,pa.array([None]*len(rows),type=pa.string()))
        with tempfile.TemporaryDirectory() as d:
            root=Path(d);pq.write_table(table,root/'source.parquet',row_group_size=3)
            entry=dict(path='synthetic',ordinal=0,sha256='synthetic')
            r=process(entry,root/'source.parquet',root/'out',batch_size=8,groups_per_unit=4,stream_groups=True)
            self.assertEqual(r['rows_confirmed'],30);self.assertEqual(r['row_groups'],10)
            r=consolidate(root/'out');self.assertEqual(r['events'],30)
            output=pa.concat_tables([pq.read_table(f) for f in r['outputs']]).to_pylist()
            self.assertEqual(len(output),1);self.assertEqual(output[0]['row_position_last'],29)
            self.assertAlmostEqual(output[0]['bid_last'],.429)
            again=process(entry,root/'source.parquet',root/'out',batch_size=5,groups_per_unit=4,stream_groups=True)
            self.assertEqual(again['rows_this_attempt'],0)

**tests/mvp/test_features.py**

In [ ]:
%%modulo tests/mvp/test_features.py
import unittest
import polars as pl
import pyarrow as pa
from polars.testing import assert_frame_equal
from polymarket_mvp.features import partial,merge

class FeaturesTests(unittest.TestCase):
    def test_batch_invariance_with_unordered_edges_and_invalid_close(self):
        rows=[]
        for i in [20,2,9,1,5,20,35]:
            rows.append(dict(asset_id='a',condition_id='c',event_ts_ms=1000+i*100,event_type='price_change',
                best_bid=.4+i*.001,best_ask=.6,bids_json=None,asks_json=None,source_archive='v2',size=1.,price=.5,side='BUY'))
        rows.append(dict(rows[-1],event_ts_ms=4999,best_bid=0.))
        table=pa.Table.from_pylist(rows)
        a=merge(pl.concat([partial(b,0,i)[0] for i,b in [(0,table.to_batches()[0])]]))
        for size in [1,2,3]:
            parts=[];start=0
            for b in table.to_batches(max_chunksize=size):
                parts.append(partial(b,0,start)[0]);start+=b.num_rows
            b=merge(pl.concat(parts))
            assert_frame_equal(a.sort(['asset_id','condition_id','second']),b.sort(['asset_id','condition_id','second']),check_column_order=False)
        self.assertEqual(a.filter(pl.col('second')==4)['quality_last'].item(),'boundary')

**tests/mvp/test_storage.py**

In [ ]:
%%modulo tests/mvp/test_storage.py
import tempfile
import unittest
import pyarrow as pa
from polymarket_mvp.storage import Manifest,digest

class PublicationTests(unittest.TestCase):
    def test_crash_retry_and_corruption(self):
        with tempfile.TemporaryDirectory() as root:
            m = Manifest(root)
            t = pa.table({'value': [1, 2]})
            with self.assertRaises(RuntimeError):
                m.publish('file:0:rg:0', {'features': t}, {'rows': 2}, 'before_commit')
            self.assertFalse(m.confirmed('file:0:rg:0'))
            self.assertEqual(len(m.orphans()), 1)
            self.assertTrue(m.publish('file:0:rg:0', {'features': t}, {'rows': 2}))
            self.assertFalse(m.publish('file:0:rg:0', {'features': t}, {'rows': 2}))
            self.assertEqual(m.verify(), 1)
            name = m.db.execute('SELECT path FROM objects').fetchone()[0]
            (m.root / name).write_bytes(b'corrupt')
            with self.assertRaises(ValueError):
                m.verify()
            m.db.close()

    def test_verified_replacement_precedes_retirement(self):
        import pyarrow.parquet as pq
        with tempfile.TemporaryDirectory() as root:
            m=Manifest(root);t=pa.table({'value':[1,2]})
            m.publish('unit',{'features':t,'relations':t},{'rows':2})
            originals=list(m.db.execute('SELECT path FROM objects'))
            replacement=m.root/'series.parquet';pq.write_table(t,replacement)
            m.compact([replacement],{'events':2,'series_rows':2,'sha256':{replacement.name:digest(replacement)}},delete_originals=False)
            self.assertTrue(all((m.root/p).exists() for (p,) in originals))
            m.db.close();m=Manifest(root);self.assertEqual(m.verify(),1)
            m.compact([replacement],{'events':2,'series_rows':2,'sha256':{replacement.name:digest(replacement)}},delete_originals=True)
            self.assertTrue(all((m.root/p).exists() for (p,) in originals if p.endswith('_relations.parquet')))
            self.assertTrue(all(not (m.root/p).exists() for (p,) in originals if p.endswith('_features.parquet')))
            replacement.write_bytes(b'corruption')
            with self.assertRaises(ValueError):m.verify()
            m.db.close()

**tests/mvp/test_windows.py**

In [ ]:
%%modulo tests/mvp/test_windows.py
import tempfile
import unittest
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from polymarket_mvp.windows import build_census,select_training,pack_windows
from polymarket_mvp.model import WindowDataset

class WindowTests(unittest.TestCase):
    def test_full_census_strict_splits_minimum_and_masks(self):
        rows=[dict(asset_id=a,condition_id='c',second=i,spread_last=.01+i%7*.001,event_ts_ms_last=i*1000,
                   file_ordinal_last=0,row_position_last=i,events=3,n_v1=0,n_v2=3) for a in ['a','b'] for i in range(1536)]
        with tempfile.TemporaryDirectory() as d:
            p=Path(d);pq.write_table(pa.Table.from_pylist(rows),p/'series.parquet')
            c=build_census([p/'series.parquet'],p/'c',{'train':(0,512),'validation':(512,1024),'test':(1024,1536)},buckets=2)
            self.assertEqual(c['split_counts'],{'train':2,'validation':2,'test':2})
            with self.assertRaises(RuntimeError):select_training(p/'c',1)
            s=select_training(p/'c',2);self.assertEqual(s['minimum_windows'],2)
            pack_windows(p/'c',p/'pack','train',True)
            w=WindowDataset(p/'pack')[0];target=w['target']
            self.assertEqual(int(w['x'][0,target:].abs().sum()),0)
            self.assertEqual(int(w['mask'][target:].sum()),0)
            self.assertEqual(int(w['input_mask'][target:target+8].sum()),8)

**Dependencias fijadas.** El texto codificado de esta celda contiene solo el archivo de versiones de las bibliotecas.

In [ ]:
guardar('README.md', '# Polymarket en Colab')
guardar("pyproject.toml", """
[project]
name = "moment-ft-polymarket"
version = "0.1.0"
description = "Auditable full-coverage Polymarket features and MOMENT LoRA"
readme = "README.md"
requires-python = ">=3.14"
dependencies = [
    "datasets>=5.1.0",
    "duckdb>=1.4",
    "huggingface-hub==2.4.0",
    "jupyterlab>=4.6.6",
    "matplotlib>=3.11.2",
    "momentfm",
    "numpy>=2.5.4",
    "pandas>=3.1.1",
    "peft==0.21.2",
    "polars>=2.1.0",
    "psutil>=7.2.2",
    "pyarrow==26.0.1",
    "scikit-learn>=1.9.1",
    "torch>=2.15.0",
    "tqdm>=4.70.1",
    "transformers==5.22.2",
]

[tool.uv.sources]
momentfm = { git = "https://github.com/moment-timeseries-foundation-model/moment.git", rev = "38f7310ad594100747ca2a8357e9c7ca7d323e0e" }

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src/polymarket_mvp"]
""".lstrip())
lock_comprimido = (
    '...'
)
guardar('uv.lock', zlib.decompress(base64.b64decode(lock_comprimido)).decode('utf-8'))

## Instalar y comprobar la tarjeta gráfica

El proyecto instala su propio Python mediante uv para mantener las versiones originales. Si CUDA no está disponible con estas bibliotecas, la celda se detiene antes de descargar los originales. La disponibilidad depende de la GPU y del controlador de la sesión de Colab.

In [ ]:
import os

def ejecutar(*args, env=None):
    subprocess.run([str(a) for a in args], cwd=PROYECTO, check=True, env=env)

uv = shutil.which('uv')
if uv is None:
    import urllib.request
    instalador = PROYECTO / 'install_uv.sh'
    urllib.request.urlretrieve('https://astral.sh/uv/install.sh', instalador)
    carpeta_uv = PROYECTO / '.uv-bin'
    ejecutar('sh', instalador, env=dict(os.environ, UV_UNMANAGED_INSTALL=str(carpeta_uv)))
    uv = str(carpeta_uv / 'uv')

ejecutar(uv, 'sync', '--frozen', '--python', '3.14')
PYTHON = PROYECTO / '.venv/bin/python'
ENTORNO = dict(os.environ, POLARS_MAX_THREADS='4', OMP_NUM_THREADS='4')

def python(*args):
    ejecutar(PYTHON, *args, env=ENTORNO)

python('-c', '''
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Se necesita una GPU CUDA compatible con BF16 en esta sesión.')
print(torch.cuda.get_device_name(0))
''')
LIBRE = shutil.disk_usage(PROYECTO).free
print(f'Espacio libre: {LIBRE / 10**9:.1f} GB')
if MODO == 'piloto' and LIBRE < 12 * 1024**3:
    raise RuntimeError('Libera al menos 12 GiB para el piloto.')
if MODO == 'integral' and not (RESULTADOS / 'mission.json').exists() and LIBRE < 900 * 10**9:
    raise RuntimeError('El modo integral nuevo requiere un disco mayor. La proyección comprobará el presupuesto exacto.')

## Comprobar las operaciones básicas

In [ ]:
python('-m', 'unittest', 'discover', '-s', 'tests/mvp', '-q')

## Ejecutar el ensayo de un día

Se usa un archivo original entero. Los resultados y el modelo se guardan en la carpeta de trabajo. Volver a lanzar con el mismo modo y rango reutiliza los resultados guardados que sigan disponibles.

In [ ]:
if not (RESULTADOS / 'preflight_full/result.json').exists():
    python('-u', 'herramientas/preflight_full.py', RESULTADOS)
python('-c', '''
from pathlib import Path
from polymarket_mvp.catalog import build_catalog
root = Path('runs/mvp/preflight_full')
if not (root / 'catalog/catalog.json').exists():
    print(build_catalog([root / 'partition'], root / 'catalog'))
''')

## Ejecutar la corrida integral

En el modo integral se leen primero los conteos internos de los 164 archivos. Después se miden los cuatro días completos que necesita la proyección. Si la proyección cabe en tiempo y disco, el coordinador completa las restantes etapas. Los resultados confirmados se reutilizan al relanzar desde la misma carpeta. Si se interrumpe el censo o el empaquetado antes de confirmar su salida, esa etapa puede repetirse.

In [ ]:
if MODO == 'integral' and not (RESULTADOS / 'full_result.json').exists():
    if not (RESULTADOS / 'inventory_enriched.json').exists():
        python('-u', 'herramientas/inventory_footers.py')
    if not (RESULTADOS / 'canonical_day_certificate.json').exists():
        python('-u', 'herramientas/certify_daily_partitioning.py')

    import json
    informe = RESULTADOS / 'parallel4_benchmark.json'
    if not informe.exists() or not json.loads(informe.read_text()).get('complete'):
        python('-u', '-c', '''
from polymarket_mvp.download import inventory
from polymarket_mvp.parallel import run_partitions
from polymarket_mvp.reporting import REPRESENTATIVE_DAYS
root = 'runs/mvp'
entries = [e for e in inventory(root)['files']
           if e['path'].split('date=')[1][:10] in REPRESENTATIVE_DAYS]
report = run_partitions(entries, root, optimized=True)
if not report['complete']:
    raise RuntimeError(report['errors'])
''')
    if not (RESULTADOS / 'projection.json').exists() or not (RESULTADOS / 'mission.json').exists():
        python('-c', f'''
from polymarket_mvp.reporting import projection
report = projection('runs/mvp', budget_hours={HORAS_PRESUPUESTO!r})
print(report)
if not report.get('performance_gate') or not report.get('storage_gate'):
    raise RuntimeError('La estimación supera los recursos de este entorno.')
''')
    python('-u', '-m', 'polymarket_mvp.full',
           '--root', RESULTADOS, '--projection', RESULTADOS / 'projection.json',
           '--budget-hours', HORAS_PRESUPUESTO)

## Consultar el modelo y sus resultados

In [ ]:
import json

ruta = RESULTADOS / ('full_result.json' if MODO == 'integral' else 'preflight_full/result.json')
resultado = json.loads(ruta.read_text())
assert resultado['complete']
print('Modo:', MODO, '| rango LoRA:', RANGO_LORA)
print('Ventanas entrenadas:', resultado['training']['trained_windows'])
for split in ('validation', 'test'):
    m = resultado[split]
    print(split, '| ventanas:', m['windows'])
    for nombre in ('persistence', 'moment', 'lora'):
        print(f"  {nombre}: MAE={m[nombre + '_mae']:.6f}  RMSE={m[nombre + '_rmse']:.6f}")
print('Resultados:', RESULTADOS)